In [222]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import seaborn as sns
import matplotlib.pyplot as plt
import sklearn
from scipy.stats.mstats import winsorize
from sklearn.model_selection import train_test_split, cross_val_score
import xgboost as xgb
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname,_, filenames in os.walk('/kaggle/input/event-recommendation-engine-dataset/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/event-recommendation-engine-dataset/event_attendees.csv
/kaggle/input/event-recommendation-engine-dataset/users.csv
/kaggle/input/event-recommendation-engine-dataset/event_popularity_benchmark_private_test_only.csv
/kaggle/input/event-recommendation-engine-dataset/public_leaderboard_solution.csv
/kaggle/input/event-recommendation-engine-dataset/event_popularity_benchmark.csv
/kaggle/input/event-recommendation-engine-dataset/user_friends.csv
/kaggle/input/event-recommendation-engine-dataset/random_benchmark.csv
/kaggle/input/event-recommendation-engine-dataset/train.csv
/kaggle/input/event-recommendation-engine-dataset/test.csv
/kaggle/input/event-recommendation-engine-dataset/events.csv


In [2]:
users_df = pd.read_csv('/kaggle/input/event-recommendation-engine-dataset/users.csv')
users_df.head(10)

,user_id,locale,birthyear,gender,joinedAt,location,timezone
0,3197468391,id_ID,1993,male,2012-10-02T06:40:55.524Z,Medan Indonesia,480.0
1,3537982273,id_ID,1992,male,2012-09-29T18:03:12.111Z,Medan Indonesia,420.0
2,823183725,en_US,1975,male,2012-10-06T03:14:07.149Z,Stratford Ontario,-240.0
3,1872223848,en_US,1991,female,2012-11-04T08:59:43.783Z,Tehran Iran,210.0
4,3429017717,id_ID,1995,female,2012-09-10T16:06:53.132Z,NaN,420.0
5,627175141,ka_GE,1973,female,2012-11-01T09:59:17.590Z,Tbilisi Georgia,240.0
6,2752000443,id_ID,1994,male,2012-10-03T05:22:17.637Z,Medan Indonesia,420.0
7,3473687777,id_ID,1965,female,2012-10-03T12:19:29.975Z,Medan Indonesia,420.0
8,2966052962,id_ID,1979,male,2012-10-31T10:11:57.668Z,Medan Indonesia,420.0
9,264876277,id_ID,1988,female,2012-10-02T07:28:09.555Z,Medan Indonesia,420.0


In [3]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38209 entries, 0 to 38208
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   user_id    38209 non-null  int64  
 1   locale     38209 non-null  object 
 2   birthyear  36717 non-null  object 
 3   gender     38100 non-null  object 
 4   joinedAt   38151 non-null  object 
 5   location   32744 non-null  object 
 6   timezone   37773 non-null  float64
dtypes: float64(1), int64(1), object(5)
memory usage: 2.0+ MB


In [4]:
users_full_df = users_df.copy()
print(len(users_df))
len(users_full_df)

38209


38209

In [5]:
events_full_df = pd.read_csv('/kaggle/input/event-recommendation-engine-dataset/events.csv')
events_full_df.head(10)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,event_id,user_id,start_time,city,state,zip,country,lat,lng,c_1,...,c_92,c_93,c_94,c_95,c_96,c_97,c_98,c_99,c_100,c_other
0,684921758,3647864012,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2,...,0,1,0,0,0,0,0,0,0,9
1,244999119,3476440521,2012-11-03T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2,...,0,0,0,0,0,0,0,0,0,7
2,3928440935,517514445,2012-11-05T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,12
3,2582345152,781585781,2012-10-30T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,1,...,0,0,0,0,0,0,0,0,0,8
4,1051165850,1016098580,2012-09-27T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,1,...,0,0,0,0,0,0,0,0,0,9
5,1212611096,1426522332,2012-11-16T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,22
6,3689283674,725266702,2012-11-02T20:00:00.003Z,NaN,NaN,NaN,NaN,NaN,NaN,0,...,0,0,0,0,0,0,0,0,0,28
7,2584113432,613687941,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,0,...,2,0,0,0,0,0,0,0,0,354
8,3365728297,1098509207,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,47.058,21.926,0,...,0,0,0,0,0,0,0,1,0,25
9,2912638473,3598071768,2012-10-18T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,1,...,0,0,0,0,0,0,0,0,0,3


In [6]:
print(len(events_full_df))
pd.Timestamp.max

3137972


Timestamp('2262-04-11 23:47:16.854775807')

In [7]:
user_friends_df = pd.read_csv('/kaggle/input/event-recommendation-engine-dataset/user_friends.csv')
user_friends_df.head(10)

,user,friends
0,3197468391,1346449342 3873244116 4226080662 1222907620 54...
1,3537982273,1491560444 395798035 2036380346 899375619 3534...
2,823183725,1484954627 1950387873 1652977611 4185960823 42...
3,1872223848,83361640 723814682 557944478 1724049724 253059...
4,3429017717,4253303705 2130310957 1838389374 3928735761 71...
5,627175141,3462311094 868148671 3475458679 1822640148 183...
6,2752000443,665103859 798664587 3773945815 595192956 31571...
7,3473687777,1481860022 3611032568 3086070277 275090520 366...
8,2966052962,4266041793 2113918851 4152864981 1055028635 12...
9,264876277,1473289379 3127593523 487736094 2990183113 249...


In [8]:
user_friends_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38202 entries, 0 to 38201
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user     38202 non-null  int64 
 1   friends  38063 non-null  object
dtypes: int64(1), object(1)
memory usage: 597.0+ KB


In [9]:
#user_friends_df['friends_list']=user_friends_df['friends'].str.split()
#user_friends_df.drop(columns='friends_list', inplace = True)
user_friends_df['friends_list']=user_friends_df.apply(lambda row:row['friends'].split() if not pd.isna(row['friends']) else [], axis =1)
user_friends_df.head(10)

,user,friends,friends_list
0,3197468391,1346449342 3873244116 4226080662 1222907620 54...,"[1346449342, 3873244116, 4226080662, 122290762..."
1,3537982273,1491560444 395798035 2036380346 899375619 3534...,"[1491560444, 395798035, 2036380346, 899375619,..."
2,823183725,1484954627 1950387873 1652977611 4185960823 42...,"[1484954627, 1950387873, 1652977611, 418596082..."
3,1872223848,83361640 723814682 557944478 1724049724 253059...,"[83361640, 723814682, 557944478, 1724049724, 2..."
4,3429017717,4253303705 2130310957 1838389374 3928735761 71...,"[4253303705, 2130310957, 1838389374, 392873576..."
5,627175141,3462311094 868148671 3475458679 1822640148 183...,"[3462311094, 868148671, 3475458679, 1822640148..."
6,2752000443,665103859 798664587 3773945815 595192956 31571...,"[665103859, 798664587, 3773945815, 595192956, ..."
7,3473687777,1481860022 3611032568 3086070277 275090520 366...,"[1481860022, 3611032568, 3086070277, 275090520..."
8,2966052962,4266041793 2113918851 4152864981 1055028635 12...,"[4266041793, 2113918851, 4152864981, 105502863..."
9,264876277,1473289379 3127593523 487736094 2990183113 249...,"[1473289379, 3127593523, 487736094, 2990183113..."


In [10]:
event_attendees_df = pd.read_csv('/kaggle/input/event-recommendation-engine-dataset/event_attendees.csv')
event_attendees_df.head(10)

,event,yes,maybe,invited,no
0,1159822043,1975964455 252302513 4226086795 3805886383 142...,2733420590 517546982 1350834692 532087573 5831...,1723091036 3795873583 4109144917 3560622906 31...,3575574655 1077296663
1,686467261,2394228942 2686116898 1056558062 3792942231 41...,1498184352 645689144 3770076778 331335845 4239...,1788073374 733302094 1830571649 676508092 7081...,NaN
2,1186208412,NaN,3320380166 3810793697,1379121209 440668682,1728988561 2950720854
3,2621578336,NaN,NaN,NaN,NaN
4,855842686,2406118796 3550897984 294255260 1125817077 109...,2671721559 1761448345 2356975806 2666669465 10...,1518670705 880919237 2326414227 2673818347 332...,3500235232
5,2018671985,NaN,NaN,NaN,NaN
6,488116622,4145960786 2550625355 2577667841 1575121941 28...,1227223575 2789471603 1323321680 3086272918 38...,1413359297 2300232602 1412759254 617751520 286...,1498160155 3708150269 823488244 3595018395 173...
7,1273761447,2680366192 2151335654 3447231284 3021641283 17...,94519567 2208454642 1749189642 2558652483 2983...,3828226993 3279641599 493535713 2091128996 298...,3215817616 101410505 3172228763 4161238910 181...
8,2688888297,298428624 2292079981 1819927116 1843127538 410...,3433056329 3166442492 1754661408 2966742619 32...,143382439 552645572 2872499486 1476024415 3890...,3433837562 492244978 784111553 3319042922
9,3870329460,NaN,4229976635,101440046 3547967849 2482041922 662878699 2600...,1200610016 379229947 1357977256 725446989


In [11]:
train= pd.read_csv('/kaggle/input/event-recommendation-engine-dataset/train.csv')
train.head(10)

,user,event,invited,timestamp,interested,not_interested
0,3044012,1918771225,0,2012-10-02 15:53:05.754000+00:00,0,0
1,3044012,1502284248,0,2012-10-02 15:53:05.754000+00:00,0,0
2,3044012,2529072432,0,2012-10-02 15:53:05.754000+00:00,1,0
3,3044012,3072478280,0,2012-10-02 15:53:05.754000+00:00,0,0
4,3044012,1390707377,0,2012-10-02 15:53:05.754000+00:00,0,0
5,3044012,1532377761,0,2012-10-02 15:53:05.754000+00:00,0,0
6,4236494,2352676247,0,2012-10-30 01:48:25.617000+00:00,0,0
7,4236494,152418051,0,2012-10-30 01:48:28.645000+00:00,1,0
8,4236494,4203627753,0,2012-10-30 01:49:14.152000+00:00,1,0
9,4236494,110357109,0,2012-10-30 01:48:25.617000+00:00,0,0


In [12]:
train_full_df = train.copy()
print(len(train))
len(train_full_df)

15398


15398

In [13]:
def info_describe(df):
    print(df.info())

In [14]:
def dataset_explode_columns(event_col,target_col,new_colname,df):
  new_df = pd.DataFrame(df[[event_col,target_col]])
  new_df[target_col] = new_df[target_col].str.split()
  new_df = new_df.explode(target_col, ignore_index=True)
  new_df.rename(columns={target_col:new_colname}, inplace = True)
  return new_df

In [15]:
event_attendees_yes_df = dataset_explode_columns('event','yes','user_id',event_attendees_df)
event_attendees_yes_df.head(10)

,event,user_id
0,1159822043,1975964455
1,1159822043,252302513
2,1159822043,4226086795
3,1159822043,3805886383
4,1159822043,1420484491
5,1159822043,3831921392
6,1159822043,3973364512
7,686467261,2394228942
8,686467261,2686116898
9,686467261,1056558062


In [16]:
event_attendees_maybe_df = dataset_explode_columns('event','maybe','user_id',event_attendees_df)
event_attendees_maybe_df.head(10)

,event,user_id
0,1159822043,2733420590
1,1159822043,517546982
2,1159822043,1350834692
3,1159822043,532087573
4,1159822043,583146976
5,1159822043,3079807774
6,1159822043,1324909047
7,686467261,1498184352
8,686467261,645689144
9,686467261,3770076778


In [17]:
event_attendees_invited_df = dataset_explode_columns('event','invited','user_id',event_attendees_df)
event_attendees_invited_df.head(10)

,event,user_id
0,1159822043,1723091036
1,1159822043,3795873583
2,1159822043,4109144917
3,1159822043,3560622906
4,1159822043,3106484834
5,1159822043,2925436522
6,1159822043,2284506787
7,1159822043,2484438140
8,1159822043,3148037960
9,1159822043,2142928184


In [18]:
event_attendees_no_df = dataset_explode_columns('event','no','user_id',event_attendees_df)
event_attendees_no_df.head(10)

,event,user_id
0,1159822043,3575574655
1,1159822043,1077296663
2,686467261,NaN
3,1186208412,1728988561
4,1186208412,2950720854
5,2621578336,NaN
6,855842686,3500235232
7,2018671985,NaN
8,488116622,1498160155
9,488116622,3708150269


In [19]:
print(len(event_attendees_yes_df))
print(len(event_attendees_maybe_df))
print(len(event_attendees_invited_df))
print(len(event_attendees_no_df))

833121
523891
9420533
481097


In [20]:
def list_sum_columns(ncols):
  col_list=[]
  for i in range(1, ncols+1):
    col_list.append('c_'+ str(i))
  col_list.append('c_other')
  print(len(col_list))
  return col_list

In [21]:
col_list = list_sum_columns(100)
print(col_list)
#events_desc_details_df = events_df[]

101
['c_1', 'c_2', 'c_3', 'c_4', 'c_5', 'c_6', 'c_7', 'c_8', 'c_9', 'c_10', 'c_11', 'c_12', 'c_13', 'c_14', 'c_15', 'c_16', 'c_17', 'c_18', 'c_19', 'c_20', 'c_21', 'c_22', 'c_23', 'c_24', 'c_25', 'c_26', 'c_27', 'c_28', 'c_29', 'c_30', 'c_31', 'c_32', 'c_33', 'c_34', 'c_35', 'c_36', 'c_37', 'c_38', 'c_39', 'c_40', 'c_41', 'c_42', 'c_43', 'c_44', 'c_45', 'c_46', 'c_47', 'c_48', 'c_49', 'c_50', 'c_51', 'c_52', 'c_53', 'c_54', 'c_55', 'c_56', 'c_57', 'c_58', 'c_59', 'c_60', 'c_61', 'c_62', 'c_63', 'c_64', 'c_65', 'c_66', 'c_67', 'c_68', 'c_69', 'c_70', 'c_71', 'c_72', 'c_73', 'c_74', 'c_75', 'c_76', 'c_77', 'c_78', 'c_79', 'c_80', 'c_81', 'c_82', 'c_83', 'c_84', 'c_85', 'c_86', 'c_87', 'c_88', 'c_89', 'c_90', 'c_91', 'c_92', 'c_93', 'c_94', 'c_95', 'c_96', 'c_97', 'c_98', 'c_99', 'c_100', 'c_other']


In [22]:
events_full_df.columns

Index(['event_id', 'user_id', 'start_time', 'city', 'state', 'zip', 'country',
       'lat', 'lng', 'c_1',
       ...
       'c_92', 'c_93', 'c_94', 'c_95', 'c_96', 'c_97', 'c_98', 'c_99', 'c_100',
       'c_other'],
      dtype='object', length=110)

In [23]:
col_desc_list = ['event_id'] + col_list
#print(col_desc_list)
events_desc_details_df = events_full_df[col_desc_list] 
events_desc_details_df.head(10)

,event_id,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,...,c_92,c_93,c_94,c_95,c_96,c_97,c_98,c_99,c_100,c_other
0,684921758,2,0,2,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,9
1,244999119,2,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,7
2,3928440935,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,12
3,2582345152,1,0,2,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,8
4,1051165850,1,1,0,0,0,0,0,2,0,...,0,0,0,0,0,0,0,0,0,9
5,1212611096,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,22
6,3689283674,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,28
7,2584113432,0,0,2,0,0,33,0,3,1,...,2,0,0,0,0,0,0,0,0,354
8,3365728297,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,25
9,2912638473,1,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,3


In [24]:
events_df = events_full_df.copy()
events_df.drop(columns = col_list, inplace = True)
events_df.head(10)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,event_id,user_id,start_time,city,state,zip,country,lat,lng
0,684921758,3647864012,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN
1,244999119,3476440521,2012-11-03T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN
2,3928440935,517514445,2012-11-05T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN
3,2582345152,781585781,2012-10-30T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN
4,1051165850,1016098580,2012-09-27T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN
5,1212611096,1426522332,2012-11-16T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN
6,3689283674,725266702,2012-11-02T20:00:00.003Z,NaN,NaN,NaN,NaN,NaN,NaN
7,2584113432,613687941,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN
8,3365728297,1098509207,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,47.058,21.926
9,2912638473,3598071768,2012-10-18T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
for col in events_df.columns:
  print("Column:", col , " Null Count:", events_df[col].isna().sum())
events_df.count()

Column: event_id  Null Count: 0
Column: user_id  Null Count: 0
Column: start_time  Null Count: 0
Column: city  Null Count: 1557125
Column: state  Null Count: 1889723
Column: zip  Null Count: 2686286
Column: country  Null Count: 1533009
Column: lat  Null Count: 1331880
Column: lng  Null Count: 1331880


event_id      3137972
user_id       3137972
start_time    3137972
city          1580847
state         1248249
zip            451686
country       1604963
lat           1806092
lng           1806092
dtype: int64

In [26]:
print(len(events_df))
print(len(events_full_df))
print(len(events_desc_details_df))

3137972
3137972
3137972


In [27]:
events_df = events_df[events_df['start_time'] <='2262-04-11 23:47:16.854775807']
len(events_df)

3137969

In [28]:
def date_conversion_func(row,col_name):
    #print(row[col_name])
    if row[col_name] is None:
        return None
    if pd.isna(row[col_name]):
        return None
    if isinstance(row[col_name],float) and np.nan(row[col_name]):
        return None
    if row[col_name].strip().lower() in ['nan', 'nat', 'null', 'none', '']:
        return None
    if row[col_name].endswith('Z'):
        date_str = row[col_name].replace('Z','+00:00')
        try:
            dt1 = datetime.fromisoformat(date_str)
            return dt1.strftime("%Y-%m-%d %H:%M:%S.%f%z")
        except ValueError:
            pass
    else:
        if len(row[col_name].strip()) == 25:
            dt1 = datetime.fromisoformat(row[col_name])
            return dt1.strftime("%Y-%m-%d %H:%M:%S.%f%z")
        else:
            return pd.to_datetime(row[col_name],format="%Y-%m-%d %H:%M:%S.%f%z", 
                                     errors='coerce', 
                                     utc=True)

In [29]:
#events_df['start_time_modified'] = events_df.apply(date_conversion_func, args = ('start_time',), axis = 1)
events_df['start_time_modified'] = events_df.apply(lambda row:date_conversion_func(row,'start_time'), axis = 1)

In [30]:
print(len(events_df[events_df['start_time_modified'].isna()==True]))
events_df.head(10)

0


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,event_id,user_id,start_time,city,state,zip,country,lat,lng,start_time_modified
0,684921758,3647864012,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-31 00:00:00.001000+0000
1,244999119,3476440521,2012-11-03T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-03 00:00:00.001000+0000
2,3928440935,517514445,2012-11-05T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-05 00:00:00.001000+0000
3,2582345152,781585781,2012-10-30T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-30 00:00:00.001000+0000
4,1051165850,1016098580,2012-09-27T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-09-27 00:00:00.001000+0000
5,1212611096,1426522332,2012-11-16T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-16 00:00:00.001000+0000
6,3689283674,725266702,2012-11-02T20:00:00.003Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-02 20:00:00.003000+0000
7,2584113432,613687941,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-31 00:00:00.001000+0000
8,3365728297,1098509207,2012-10-31T00:00:00.001Z,NaN,NaN,NaN,NaN,47.058,21.926,2012-10-31 00:00:00.001000+0000
9,2912638473,3598071768,2012-10-18T00:00:00.001Z,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-18 00:00:00.001000+0000


In [31]:
events_df.isna().sum()

event_id                     0
user_id                      0
start_time                   0
city                   1557122
state                  1889720
zip                    2686283
country                1533006
lat                    1331877
lng                    1331877
start_time_modified          0
dtype: int64

In [32]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38209 entries, 0 to 38208
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   user_id    38209 non-null  int64  
 1   locale     38209 non-null  object 
 2   birthyear  36717 non-null  object 
 3   gender     38100 non-null  object 
 4   joinedAt   38151 non-null  object 
 5   location   32744 non-null  object 
 6   timezone   37773 non-null  float64
dtypes: float64(1), int64(1), object(5)
memory usage: 2.0+ MB


In [33]:
users_df['joinedAt_modified'] = users_df.apply(date_conversion_func, args = ('joinedAt',), axis = 1)

In [34]:
print(users_df.isna().sum())
print(users_df['joinedAt'].isna().sum())
print(users_df['joinedAt_modified'].isna().sum())

user_id                 0
locale                  0
birthyear            1492
gender                109
joinedAt               58
location             5465
timezone              436
joinedAt_modified      58
dtype: int64
58
58


In [35]:
from datetime import timedelta

In [36]:
events_df.drop(columns = ['start_time'], inplace=True)
events_df.rename(columns = {'start_time_modified':'start_time'}, inplace = True)
events_df.head(10)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,event_id,user_id,city,state,zip,country,lat,lng,start_time
0,684921758,3647864012,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-31 00:00:00.001000+0000
1,244999119,3476440521,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-03 00:00:00.001000+0000
2,3928440935,517514445,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-05 00:00:00.001000+0000
3,2582345152,781585781,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-30 00:00:00.001000+0000
4,1051165850,1016098580,NaN,NaN,NaN,NaN,NaN,NaN,2012-09-27 00:00:00.001000+0000
5,1212611096,1426522332,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-16 00:00:00.001000+0000
6,3689283674,725266702,NaN,NaN,NaN,NaN,NaN,NaN,2012-11-02 20:00:00.003000+0000
7,2584113432,613687941,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-31 00:00:00.001000+0000
8,3365728297,1098509207,NaN,NaN,NaN,NaN,47.058,21.926,2012-10-31 00:00:00.001000+0000
9,2912638473,3598071768,NaN,NaN,NaN,NaN,NaN,NaN,2012-10-18 00:00:00.001000+0000


In [37]:
train['timestamp_modified'] = train.apply(date_conversion_func, args = ('timestamp',), axis = 1)
train.head(10)

,user,event,invited,timestamp,interested,not_interested,timestamp_modified
0,3044012,1918771225,0,2012-10-02 15:53:05.754000+00:00,0,0,2012-10-02 15:53:05.754000+00:00
1,3044012,1502284248,0,2012-10-02 15:53:05.754000+00:00,0,0,2012-10-02 15:53:05.754000+00:00
2,3044012,2529072432,0,2012-10-02 15:53:05.754000+00:00,1,0,2012-10-02 15:53:05.754000+00:00
3,3044012,3072478280,0,2012-10-02 15:53:05.754000+00:00,0,0,2012-10-02 15:53:05.754000+00:00
4,3044012,1390707377,0,2012-10-02 15:53:05.754000+00:00,0,0,2012-10-02 15:53:05.754000+00:00
5,3044012,1532377761,0,2012-10-02 15:53:05.754000+00:00,0,0,2012-10-02 15:53:05.754000+00:00
6,4236494,2352676247,0,2012-10-30 01:48:25.617000+00:00,0,0,2012-10-30 01:48:25.617000+00:00
7,4236494,152418051,0,2012-10-30 01:48:28.645000+00:00,1,0,2012-10-30 01:48:28.645000+00:00
8,4236494,4203627753,0,2012-10-30 01:49:14.152000+00:00,1,0,2012-10-30 01:49:14.152000+00:00
9,4236494,110357109,0,2012-10-30 01:48:25.617000+00:00,0,0,2012-10-30 01:48:25.617000+00:00


In [38]:
train.isna().sum()

user                  0
event                 0
invited               0
timestamp             0
interested            0
not_interested        0
timestamp_modified    0
dtype: int64

In [39]:
users_df['joinedAt_modified'] = pd.to_datetime(users_df['joinedAt_modified']) + pd.to_timedelta(users_df['timezone'].fillna(0),unit = 'm')
users_df.head(10)

,user_id,locale,birthyear,gender,joinedAt,location,timezone,joinedAt_modified
0,3197468391,id_ID,1993,male,2012-10-02T06:40:55.524Z,Medan Indonesia,480.0,2012-10-02 14:40:55.524000+00:00
1,3537982273,id_ID,1992,male,2012-09-29T18:03:12.111Z,Medan Indonesia,420.0,2012-09-30 01:03:12.111000+00:00
2,823183725,en_US,1975,male,2012-10-06T03:14:07.149Z,Stratford Ontario,-240.0,2012-10-05 23:14:07.149000+00:00
3,1872223848,en_US,1991,female,2012-11-04T08:59:43.783Z,Tehran Iran,210.0,2012-11-04 12:29:43.783000+00:00
4,3429017717,id_ID,1995,female,2012-09-10T16:06:53.132Z,NaN,420.0,2012-09-10 23:06:53.132000+00:00
5,627175141,ka_GE,1973,female,2012-11-01T09:59:17.590Z,Tbilisi Georgia,240.0,2012-11-01 13:59:17.590000+00:00
6,2752000443,id_ID,1994,male,2012-10-03T05:22:17.637Z,Medan Indonesia,420.0,2012-10-03 12:22:17.637000+00:00
7,3473687777,id_ID,1965,female,2012-10-03T12:19:29.975Z,Medan Indonesia,420.0,2012-10-03 19:19:29.975000+00:00
8,2966052962,id_ID,1979,male,2012-10-31T10:11:57.668Z,Medan Indonesia,420.0,2012-10-31 17:11:57.668000+00:00
9,264876277,id_ID,1988,female,2012-10-02T07:28:09.555Z,Medan Indonesia,420.0,2012-10-02 14:28:09.555000+00:00


In [40]:
users_df.isna().sum()

user_id                 0
locale                  0
birthyear            1492
gender                109
joinedAt               58
location             5465
timezone              436
joinedAt_modified      58
dtype: int64

In [41]:
users_df.drop(columns = 'joinedAt', inplace = True)
users_df.rename(columns = {'joinedAt_modified':'joinedAt'}, inplace = True)
users_df.head(10)

,user_id,locale,birthyear,gender,location,timezone,joinedAt
0,3197468391,id_ID,1993,male,Medan Indonesia,480.0,2012-10-02 14:40:55.524000+00:00
1,3537982273,id_ID,1992,male,Medan Indonesia,420.0,2012-09-30 01:03:12.111000+00:00
2,823183725,en_US,1975,male,Stratford Ontario,-240.0,2012-10-05 23:14:07.149000+00:00
3,1872223848,en_US,1991,female,Tehran Iran,210.0,2012-11-04 12:29:43.783000+00:00
4,3429017717,id_ID,1995,female,NaN,420.0,2012-09-10 23:06:53.132000+00:00
5,627175141,ka_GE,1973,female,Tbilisi Georgia,240.0,2012-11-01 13:59:17.590000+00:00
6,2752000443,id_ID,1994,male,Medan Indonesia,420.0,2012-10-03 12:22:17.637000+00:00
7,3473687777,id_ID,1965,female,Medan Indonesia,420.0,2012-10-03 19:19:29.975000+00:00
8,2966052962,id_ID,1979,male,Medan Indonesia,420.0,2012-10-31 17:11:57.668000+00:00
9,264876277,id_ID,1988,female,Medan Indonesia,420.0,2012-10-02 14:28:09.555000+00:00


In [42]:
locale_id_country_dict = {'id_ID': 'Indonesia','en_US': 'United States','ka_GE': 'Georgia','es_LA': 'Spanish Latin America','fr_FR': 'France',
                          'ar_AR': 'Arab','en_GB': 'United Kingdom' ,'pt_BR': 'Brazil','th_TH': 'Thailand','vi_VN': 'Vietnam','fa_IR': 'Iran',
                          'es_ES': 'Spain','hu_HU': 'Hungary','cs_CZ': 'Czech','pt_PT': 'Portugal','bs_BA': 'Bosnia','ko_KR': 'Korean','ru_RU': 'Russia', 
                          'zh_TW': 'China', 'de_DE': 'Germany','mn_MN': 'Mongolia',      'zh_CN': 'China', 'ja_JP': 'Japan', 'bg_BG': 'Bulgaria', 
                          'km_KH': 'Cambodia','it_IT': 'Italy','fr_CA': 'Canada','fi_FI': 'Finland','tr_TR': 'Turkey', 'sq_AL': 'Albania', 
                          'zh_HK': 'Hong Kong China', 'el_GR': 'Greece','jv_ID': 'Indonesia','ro_RO': 'Romania','sk_SK': 'Slovakia', 'pl_PL': 'Poland',
                          'ms_MY': 'Malaysia','az_AZ': 'Azerbaijan','nl_NL': 'Netherland','hr_HR': 'Croatia','en_IN': 'India','hi_IN': 'India',
                          'ca_ES': 'Spain','da_DK': 'Denmark','fb_LT': 'Los Angeles','uk_UA': 'Ukraine','nb_NO': 'Norway','sv_SE': 'Sweden',
                          'et_EE': 'Estonia','bn_IN': 'India','sr_RS': 'Serbia','lt_LT': 'Switzerland','pa_IN': 'India','af_ZA': 'South Africa',
                          'lv_LV': 'Latvia','tl_PH': 'Philippines','eo_EO': 'Spain','he_IL': 'Hebrew','mk_MK': 'Macedonia','ku_TR': 'Turkey',
                          'es_MX': 'Mexico' ,'cy_GB': 'United Kingdom'}
print(len(locale_id_country_dict))

62


In [43]:
users_df.loc[(users_df['location'].isna() ==False) & (users_df['location'].str.contains('undefined'))]

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user_id,locale,birthyear,gender,location,timezone,joinedAt
164,3622381291,en_US,1985,male,undefined undefined,240.0,2012-10-22 20:28:56.082000+00:00
553,535307877,en_US,1990,male,undefined undefined,-660.0,2012-10-31 14:27:51.377000+00:00
891,3408917707,en_US,1990,male,undefined undefined,390.0,2012-09-19 21:44:34.434000+00:00
1070,1809024609,en_US,1993,male,undefined undefined,480.0,2012-11-12 00:33:49.306000+00:00
1122,4263441346,en_US,1992,male,undefined undefined,480.0,2012-11-01 00:01:17.050000+00:00
...,...,...,...,...,...,...,...
36704,215063706,en_US,1992,male,undefined undefined,NaN,2012-11-04 07:11:53.083000+00:00
36714,930683638,en_US,1992,male,undefined undefined,480.0,2012-11-13 07:22:37.696000+00:00
37281,1555049961,en_US,1993,male,undefined undefined,330.0,2012-10-13 22:07:01.086000+00:00
37450,4058033929,id_ID,1989,male,undefined undefined,420.0,2012-10-28 16:17:19.624000+00:00


In [44]:
def fill_user_location(row):
    try:
        if 'undefined' in str(row['location']) or pd.isna(row['location']):
            if row['locale'] in locale_id_country_dict.keys():
                return locale_id_country_dict[row['locale']]
            else:
                return ""
        else:
            return row['location']
    except Exception as e:
        print(f"Error occured while processing row for user_id {row['user_id']}. Error Details: {e}")
        return ""
        
            

In [45]:
print(len(users_df[users_df['location'].isna()])) #5465
print(len(users_df[(users_df['location'].str.contains('undefined')) & (users_df['location'].isna()==False)])) #191
users_df.loc[:,'location'] = users_df.apply(lambda row: fill_user_location(row) , axis = 1)
print(len(users_df[users_df['location'].isna()])) #5465
print(len(users_df[(users_df['location'].str.contains('undefined')) & (users_df['location'].isna()==False)])) #191


5465
191
0
0


In [46]:
train.drop(columns = 'timestamp', inplace = True)
train.rename(columns = {'timestamp_modified':'timestamp'},inplace= True)
#train[train['timestamp'].str.len()==25]
train.head(10)

,user,event,invited,interested,not_interested,timestamp
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00


In [47]:
print(len(event_attendees_yes_df))
print(len(event_attendees_maybe_df))
print(len(event_attendees_no_df))
print(len(event_attendees_invited_df))

833121
523891
481097
9420533


In [48]:
event_attendees_yes_df['user_id'] = event_attendees_yes_df['user_id'].astype("float").astype("Int64")
event_attendees_maybe_df['user_id'] = event_attendees_maybe_df['user_id'].astype("float").astype("Int64")
event_attendees_invited_df['user_id'] = event_attendees_invited_df['user_id'].astype("float").astype("Int64")
event_attendees_no_df['user_id'] = event_attendees_no_df['user_id'].astype("float").astype("Int64")

In [49]:
event_attendees_invited_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9420533 entries, 0 to 9420532
Data columns (total 2 columns):
 #   Column   Dtype
---  ------   -----
 0   event    int64
 1   user_id  Int64
dtypes: Int64(1), int64(1)
memory usage: 152.7 MB


In [50]:
event_user_response_df = pd.concat([event_attendees_yes_df,event_attendees_maybe_df,event_attendees_no_df])
print(len(event_user_response_df))
event_user_response_df=event_user_response_df.drop_duplicates()
print(len(event_user_response_df))

1838109
1833576


In [51]:
event_user_response_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1833576 entries, 0 to 481096
Data columns (total 2 columns):
 #   Column   Dtype
---  ------   -----
 0   event    int64
 1   user_id  Int64
dtypes: Int64(1), int64(1)
memory usage: 43.7 MB


In [52]:
print(len(event_user_response_df))
event_user_response_df = event_user_response_df.merge(events_df[['event_id','start_time']],how = 'left',left_on='event',right_on='event_id')
event_user_response_df.count()

1833576


event         1833576
user_id       1826250
event_id      1810048
start_time    1810048
dtype: int64

In [53]:
event_user_response_df.drop(columns = ['event_id'], inplace = True)
#event_user_response_df[event_user_response_df['event_id'].isna()==True]

In [54]:
event_user_response_df.head(10)

,event,user_id,start_time
0,1159822043,1975964455,2012-12-16 09:00:00.003000+0000
1,1159822043,252302513,2012-12-16 09:00:00.003000+0000
2,1159822043,4226086795,2012-12-16 09:00:00.003000+0000
3,1159822043,3805886383,2012-12-16 09:00:00.003000+0000
4,1159822043,1420484491,2012-12-16 09:00:00.003000+0000
5,1159822043,3831921392,2012-12-16 09:00:00.003000+0000
6,1159822043,3973364512,2012-12-16 09:00:00.003000+0000
7,686467261,2394228942,2012-11-23 03:00:00.003000+0000
8,686467261,2686116898,2012-11-23 03:00:00.003000+0000
9,686467261,1056558062,2012-11-23 03:00:00.003000+0000


In [55]:
train_user = pd.DataFrame(train['user'].unique(), columns = ['user'])
train_user.head(10)

,user
0,3044012
1,4236494
2,5574997
3,7547671
4,10329108
5,13301498
6,14411090
7,15469418
8,16210231
9,19283444


In [56]:
#event_user_remaining_time_df = event_user_response_df.merge(train_user, how= 'inner', left_on = 'event', right_on= 'event')
event_user_response_df = event_user_response_df.merge(users_df[['user_id','joinedAt']], how= 'left', left_on = 'user_id', right_on= 'user_id')
print(len(event_user_response_df))
event_user_response_df.head(10)

1833576


,event,user_id,start_time,joinedAt
0,1159822043,1975964455,2012-12-16 09:00:00.003000+0000,NaT
1,1159822043,252302513,2012-12-16 09:00:00.003000+0000,NaT
2,1159822043,4226086795,2012-12-16 09:00:00.003000+0000,NaT
3,1159822043,3805886383,2012-12-16 09:00:00.003000+0000,NaT
4,1159822043,1420484491,2012-12-16 09:00:00.003000+0000,NaT
5,1159822043,3831921392,2012-12-16 09:00:00.003000+0000,NaT
6,1159822043,3973364512,2012-12-16 09:00:00.003000+0000,NaT
7,686467261,2394228942,2012-11-23 03:00:00.003000+0000,NaT
8,686467261,2686116898,2012-11-23 03:00:00.003000+0000,NaT
9,686467261,1056558062,2012-11-23 03:00:00.003000+0000,NaT


In [57]:
event_user_response_df.isna().sum()

event               0
user_id          7326
start_time      23528
joinedAt      1818561
dtype: int64

In [58]:
event_user_response_df['hrs_join_event_remaining_time'] = (pd.to_datetime(event_user_response_df['start_time'])- 
                                                                         pd.to_datetime(event_user_response_df['joinedAt'])).dt.total_seconds()/3600
event_user_response_df.head(10)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,event,user_id,start_time,joinedAt,hrs_join_event_remaining_time
0,1159822043,1975964455,2012-12-16 09:00:00.003000+0000,NaT,NaN
1,1159822043,252302513,2012-12-16 09:00:00.003000+0000,NaT,NaN
2,1159822043,4226086795,2012-12-16 09:00:00.003000+0000,NaT,NaN
3,1159822043,3805886383,2012-12-16 09:00:00.003000+0000,NaT,NaN
4,1159822043,1420484491,2012-12-16 09:00:00.003000+0000,NaT,NaN
5,1159822043,3831921392,2012-12-16 09:00:00.003000+0000,NaT,NaN
6,1159822043,3973364512,2012-12-16 09:00:00.003000+0000,NaT,NaN
7,686467261,2394228942,2012-11-23 03:00:00.003000+0000,NaT,NaN
8,686467261,2686116898,2012-11-23 03:00:00.003000+0000,NaT,NaN
9,686467261,1056558062,2012-11-23 03:00:00.003000+0000,NaT,NaN


In [59]:
event_user_response_df[event_user_response_df['joinedAt'].isna()==False]

,event,user_id,start_time,joinedAt,hrs_join_event_remaining_time
143,3041357942,2835648396,2012-12-16 02:00:00.003000+0000,2012-08-29 17:24:54.715000+00:00,2600.584802
1061,1300748131,499852578,2012-12-15 11:00:00.003000+0000,2012-10-27 13:09:20.355000+00:00,1173.844347
1069,1300748131,4090605042,2012-12-15 11:00:00.003000+0000,2012-10-01 10:51:25.271000+00:00,1800.142981
1143,2971099513,2693391182,2012-12-14 03:00:00.003000+0000,2012-10-04 18:11:13.442000+00:00,1688.812934
1247,61477903,1541194342,2012-08-03 19:30:00.000000+0000,2012-06-07 16:04:51.389000+00:00,1371.419059
...,...,...,...,...,...
1832547,3217162477,2553815479,2012-09-22 17:00:00.003000+0000,2012-07-10 23:43:23.776000+00:00,1769.276730
1832745,1841631859,3303736189,2012-11-23 00:00:00.001000+0000,2012-09-27 18:35:37.284000+00:00,1349.406310
1833006,3169321620,1017683371,2012-07-21 17:00:00.000000+0000,2012-10-29 21:24:06.730000+00:00,-2404.401869
1833368,439904567,458389001,2012-08-20 02:00:00.003000+0000,2012-07-24 22:41:53.799000+00:00,627.301723


In [60]:
event_train_user_time_data = event_user_response_df.merge(train_user, how = 'inner', left_on='user_id', right_on='user')
event_train_user_time_data.head(10)

,event,user_id,start_time,joinedAt,hrs_join_event_remaining_time,user
0,2370583755,338395830,2012-12-14 04:00:00.003000+0000,2012-11-21 19:45:55.844000+00:00,536.234489,338395830
1,1389928885,2500150280,2012-12-12 03:00:00.003000+0000,2012-09-21 12:04:48.380000+00:00,1958.919895,2500150280
2,3753704467,3118355226,2012-12-12 12:00:00.002000+0000,2012-10-26 20:43:42.261000+00:00,1119.271595,3118355226
3,235476646,2768571751,2012-11-12 02:00:00.003000+0000,2012-10-19 12:50:31.917000+00:00,565.157802,2768571751
4,606080367,3621549603,2012-12-14 03:00:00.003000+0000,2012-07-31 12:32:32.552000+00:00,3254.457625,3621549603
5,444507187,2713002362,2012-09-23 00:00:00.001000+0000,2012-07-25 17:37:05.723000+00:00,1422.381744,2713002362
6,444507187,4244109605,2012-09-23 00:00:00.001000+0000,2012-09-25 10:24:21.456000+00:00,-58.405960,4244109605
7,401945277,1912059312,2012-12-02 05:00:00.003000+0000,2012-11-01 13:30:04.513000+00:00,735.498747,1912059312
8,3208056309,743008535,2012-08-31 23:00:00.003000+0000,2012-10-09 02:01:32.306000+00:00,-915.025640,743008535
9,416330287,3725766398,2012-11-11 04:00:00.003000+0000,2012-09-10 21:50:22.406000+00:00,1470.160444,3725766398


In [61]:
event_train_user_time_data.drop(columns = 'user',inplace= True)
event_train_user_time_data[event_train_user_time_data['hrs_join_event_remaining_time'].isna()==True]

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,event,user_id,start_time,joinedAt,hrs_join_event_remaining_time
150,1321395216,2008761267,NaN,2012-10-03 18:27:21.917000+00:00,NaN
779,2494370417,1720019070,NaN,2012-10-30 23:33:03.986000+00:00,NaN
780,2494370417,528289771,NaN,2012-10-29 17:59:55.390000+00:00,NaN
781,2494370417,1541564886,NaN,2012-10-30 19:21:32.531000+00:00,NaN
782,2494370417,3838242688,NaN,2012-11-01 08:36:08.087000+00:00,NaN
783,2494370417,3500240032,NaN,2012-11-01 15:02:26.057000+00:00,NaN
784,2494370417,2184501135,NaN,2012-11-03 23:46:41.998000+00:00,NaN
785,2494370417,1037260580,NaN,2012-11-01 13:41:21.127000+00:00,NaN
786,2494370417,3511417249,NaN,2012-11-01 13:12:48.423000+00:00,NaN
787,2494370417,1127657981,NaN,2012-11-04 09:37:28.703000+00:00,NaN


In [62]:
df = pd.merge(train,events_df[['event_id','user_id','start_time','city','state','zip','country']], left_on =['event'],right_on=['event_id'] ,how = 'left')
df.rename(columns={'city':'event_city','state':'event_state','zip':'event_zip','country':'event_country','start_time':'event_start_time'},inplace=True)
df.drop(columns=['event_id','user_id'],inplace=True)
print(len(df))
print(len(train))
df.head(10)

15398
15398


,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,event_country
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,NaN
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,Indonesia
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,Indonesia
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,NaN
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,Indonesia
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,Indonesia
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,NaN
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,Mauritius
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,NaN
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,NaN


In [63]:
df = df.merge(users_df,left_on=['user'],right_on=['user_id'],how='left')
df.drop(columns=['user_id',	'locale','timezone'],inplace=True)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,event_country,birthyear,gender,location,joinedAt
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,NaN,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,NaN,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,Mauritius,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00


In [64]:
'''df['is_same_country']= df.apply(lambda row:1 if row['event_country']== row['country'] else 0, axis = 1)
df['is_same_city']= df.apply(lambda row:1 if row['event_city']== row['city'] else 0, axis = 1)
df.head(10)'''

"df['is_same_country']= df.apply(lambda row:1 if row['event_country']== row['country'] else 0, axis = 1)\ndf['is_same_city']= df.apply(lambda row:1 if row['event_city']== row['city'] else 0, axis = 1)\ndf.head(10)"

In [65]:
def location_related_features(row):
    match_list = []
    loc_set = set(row['location'].split())
    if not pd.isna(row['event_city']):
        count = len(loc_set.intersection(set(row['event_city'].split())))
        if count >= 1:
            match_list.append('city')
    if not pd.isna(row['event_state']):
        count = len(loc_set.intersection(set(row['event_state'].split())))
        if count >= 1:
            match_list.append('state')
    if not pd.isna(row['event_zip']):
        count = len(loc_set.intersection(set(row['event_zip'].split())))
        if count >= 1:
            match_list.append('zip')
    if not pd.isna(row['event_country']):
        count = len(loc_set.intersection(set(row['event_country'].split())))
        if count >= 1:
            match_list.append('country')
    #print(match_list)
    return match_list

In [66]:
df['loc_match_col'] = df.apply(lambda row:location_related_features(row) if not pd.isna(row['location']) else [], axis = 1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,event_country,birthyear,gender,location,joinedAt,loc_match_col
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,NaN,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[]
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[]
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[]
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,NaN,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[]
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[]
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[]
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[]
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,Mauritius,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,"[city, country]"
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[]
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[]


In [67]:
df['is_same_city'] = df.apply(lambda row:1 if 'city' in row['loc_match_col'] else 0, axis = 1)
df['is_same_country'] = df.apply(lambda row:1 if 'country' in row['loc_match_col'] else 0, axis = 1)
df['is_same_state'] = df.apply(lambda row:1 if 'state' in row['loc_match_col'] else 0, axis = 1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,event_country,birthyear,gender,location,joinedAt,loc_match_col,is_same_city,is_same_country,is_same_state
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,NaN,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,NaN,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,Indonesia,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,Mauritius,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,"[city, country]",1,1,0
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,NaN,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0


In [68]:
df['hrs_to_event'] = (pd.to_datetime(df['event_start_time']) - pd.to_datetime(df['timestamp'])).dt.total_seconds()/3600
df['hrs_join_to_event'] = (pd.to_datetime(df['event_start_time']) - pd.to_datetime(df['joinedAt'])).dt.total_seconds()/3600
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,birthyear,gender,location,joinedAt,loc_match_col,is_same_city,is_same_country,is_same_state,hrs_to_event,hrs_join_to_event
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,16.115069,8.160827
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,19.115069,11.160828
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,573.615069,565.660828
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,85.115069,77.160828
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,83.115069,75.160828
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,1990,male,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,85.115069,77.160828
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0,118.192884,114.211470
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,"[city, country]",1,1,0,101.192044,97.211471
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0,22.179403,18.211470
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,1998,female,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0,-1.807116,-5.788530


In [69]:
df.loc[:,'hrs_to_event'] = df.apply(lambda row:0 if pd.to_datetime(row['event_start_time']) < pd.to_datetime(row['timestamp'])
                              else ((pd.to_datetime(row['event_start_time']) - pd.to_datetime(row['timestamp'])).total_seconds()/3600), axis= 1)
df.loc[:,'hrs_join_to_event'] = df.apply(lambda row:0 if pd.to_datetime(row['event_start_time']) < pd.to_datetime(row['joinedAt'])
                                   else (pd.to_datetime(row['event_start_time']) - pd.to_datetime(row['joinedAt'])).total_seconds()/3600, axis =1)
print(len(df[df['hrs_join_to_event']<0]))
print(len(df[df['hrs_to_event']<0]))
df[['event','user','hrs_to_event','hrs_join_to_event']].head(10)

0
0


,event,user,hrs_to_event,hrs_join_to_event
0,1918771225,3044012,16.115069,8.160827
1,1502284248,3044012,19.115069,11.160828
2,2529072432,3044012,573.615069,565.660828
3,3072478280,3044012,85.115069,77.160828
4,1390707377,3044012,83.115069,75.160828
5,1532377761,3044012,85.115069,77.160828
6,2352676247,4236494,118.192884,114.211470
7,152418051,4236494,101.192044,97.211471
8,4203627753,4236494,22.179403,18.211470
9,110357109,4236494,0.000000,0.000000


In [70]:
df[df['hrs_to_event']<0]

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,birthyear,gender,location,joinedAt,loc_match_col,is_same_city,is_same_country,is_same_state,hrs_to_event,hrs_join_to_event


In [71]:
def get_hrs_to_event_category(row,col_name):
    if row[col_name]<=0:
        return 0
    if row[col_name]>=1 and row[col_name]<=2:
        return 1
    if row[col_name]>=2 and row[col_name]<=4:
        return 2
    if row[col_name]>4 and row[col_name]<=6:
        return 3
    if row[col_name]>6 and row[col_name]<=12:
        return 4
    if row[col_name]>12 and row[col_name]<=24:
        return 5
    if row[col_name]>24 and row[col_name]<=(24*3):
        return 6
    if row[col_name]>(24*3) and row[col_name]<=(24*7):
        return 7
    if row[col_name]>(24*7):
        return 8

In [72]:
df['hrs_to_event_category'] = df.apply(get_hrs_to_event_category,args=('hrs_to_event',),axis = 1)
df['hrs_join_to_event_category'] = df.apply(get_hrs_to_event_category,args=('hrs_join_to_event',),axis = 1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,location,joinedAt,loc_match_col,is_same_city,is_same_country,is_same_state,hrs_to_event,hrs_join_to_event,hrs_to_event_category,hrs_join_to_event_category
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,16.115069,8.160827,5.0,4.0
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,19.115069,11.160828,5.0,4.0
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,573.615069,565.660828,8.0,8.0
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,85.115069,77.160828,7.0,7.0
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,83.115069,75.160828,7.0,7.0
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,Binjai,2012-10-02 23:50:21.023000+00:00,[],0,0,0,85.115069,77.160828,7.0,7.0
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0,118.192884,114.211470,7.0,7.0
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,"[city, country]",1,1,0,101.192044,97.211471,7.0,7.0
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0,22.179403,18.211470,5.0,5.0
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,Beau Vallon Grand Port Mauritius,2012-10-30 05:47:18.708000+00:00,[],0,0,0,0.000000,0.000000,0.0,0.0


In [73]:
def get_age_category(row,col_name):
    if row[col_name]<=12:
        return 'child'
    if row[col_name]>12 and row[col_name]<=19:
        return 'teen'
    if row[col_name]>19 and row[col_name]<30:
        return 'adult'
    if row[col_name]>=30 and row[col_name]<=45:
        return 'working adults'
    if row[col_name]>45 and row[col_name]<=60:
        return 'senior working adults'
    if row[col_name]>60:
        return 'elder'

In [74]:
df['Age'] = pd.to_datetime(df['event_start_time']).dt.year - df['birthyear'].fillna(0).astype(int)
df['Age_Category'] = df.apply(get_age_category, args = ('Age',), axis = 1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,loc_match_col,is_same_city,is_same_country,is_same_state,hrs_to_event,hrs_join_to_event,hrs_to_event_category,hrs_join_to_event_category,Age,Age_Category
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,[],0,0,0,16.115069,8.160827,5.0,4.0,22,adult
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,[],0,0,0,19.115069,11.160828,5.0,4.0,22,adult
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,[],0,0,0,573.615069,565.660828,8.0,8.0,22,adult
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,[],0,0,0,85.115069,77.160828,7.0,7.0,22,adult
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,[],0,0,0,83.115069,75.160828,7.0,7.0,22,adult
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,[],0,0,0,85.115069,77.160828,7.0,7.0,22,adult
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,[],0,0,0,118.192884,114.211470,7.0,7.0,14,teen
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,"[city, country]",1,1,0,101.192044,97.211471,7.0,7.0,14,teen
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,[],0,0,0,22.179403,18.211470,5.0,5.0,14,teen
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,[],0,0,0,0.000000,0.000000,0.0,0.0,14,teen


In [75]:
event_train_user_time_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2031 entries, 0 to 2030
Data columns (total 5 columns):
 #   Column                         Non-Null Count  Dtype              
---  ------                         --------------  -----              
 0   event                          2031 non-null   int64              
 1   user_id                        2031 non-null   Int64              
 2   start_time                     2014 non-null   object             
 3   joinedAt                       2031 non-null   datetime64[ns, UTC]
 4   hrs_join_event_remaining_time  2014 non-null   float64            
dtypes: Int64(1), datetime64[ns, UTC](1), float64(1), int64(1), object(1)
memory usage: 81.4+ KB


In [76]:
event_train_user_time_data_df = pd.DataFrame(event_train_user_time_data.groupby(['user_id']).mean(['hrs_join_event_remaining_time'])).reset_index()
event_train_user_time_data_df.head(10)

,user_id,event,hrs_join_event_remaining_time
0,10329108,2.415874e+09,589.922435
1,13301498,2.749914e+09,569.569594
2,19283444,4.439719e+08,132.902601
3,20041353,2.119689e+09,210.216911
4,24978365,2.435375e+09,295.621726
5,28806872,3.509459e+08,1321.618874
6,33966018,3.387226e+09,1844.552302
7,43026617,1.707508e+09,285.978857
8,65801702,2.290667e+09,1907.156735
9,69302861,3.113260e+09,2426.748744


In [77]:
event_train_user_time_data_df.drop(columns = 'event',inplace = True)
event_train_user_time_data_df.rename(columns = {'hrs_join_event_remaining_time':'avg_hrs_join_event_remaining_time'},inplace = True)
event_train_user_time_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 479 entries, 0 to 478
Data columns (total 2 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   user_id                            479 non-null    Int64  
 1   avg_hrs_join_event_remaining_time  479 non-null    float64
dtypes: Int64(1), float64(1)
memory usage: 8.1 KB


In [78]:
print(len(df))
df.info()

15398
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15398 entries, 0 to 15397
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   user                        15398 non-null  int64              
 1   event                       15398 non-null  int64              
 2   invited                     15398 non-null  int64              
 3   interested                  15398 non-null  int64              
 4   not_interested              15398 non-null  int64              
 5   timestamp                   15398 non-null  object             
 6   event_start_time            15398 non-null  object             
 7   event_city                  7333 non-null   object             
 8   event_state                 3275 non-null   object             
 9   event_zip                   1051 non-null   object             
 10  event_country               7361 non-null   object  

In [79]:
df = df.merge(event_train_user_time_data_df,how = 'left', left_on = 'user', right_on='user_id')
print(len(df))
df.head(10)

15398


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,is_same_country,is_same_state,hrs_to_event,hrs_join_to_event,hrs_to_event_category,hrs_join_to_event_category,Age,Age_Category,user_id,avg_hrs_join_event_remaining_time
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,0,0,16.115069,8.160827,5.0,4.0,22,adult,<NA>,NaN
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,0,0,19.115069,11.160828,5.0,4.0,22,adult,<NA>,NaN
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,0,0,573.615069,565.660828,8.0,8.0,22,adult,<NA>,NaN
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,0,0,85.115069,77.160828,7.0,7.0,22,adult,<NA>,NaN
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,0,0,83.115069,75.160828,7.0,7.0,22,adult,<NA>,NaN
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,0,0,85.115069,77.160828,7.0,7.0,22,adult,<NA>,NaN
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,118.192884,114.211470,7.0,7.0,14,teen,<NA>,NaN
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,1,0,101.192044,97.211471,7.0,7.0,14,teen,<NA>,NaN
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,22.179403,18.211470,5.0,5.0,14,teen,<NA>,NaN
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,0.000000,0.000000,0.0,0.0,14,teen,<NA>,NaN


In [80]:
df.drop(columns = 'user_id',inplace=True)

In [81]:
df[df['avg_hrs_join_event_remaining_time'].isna() == False]

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,is_same_city,is_same_country,is_same_state,hrs_to_event,hrs_join_to_event,hrs_to_event_category,hrs_join_to_event_category,Age,Age_Category,avg_hrs_join_event_remaining_time
25,10329108,1287318780,0,0,0,2012-11-07 03:05:24.446000+00:00,2012-11-11 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,0,92.909876,85.922435,7.0,7.0,20,adult,589.922435
26,10329108,600752472,0,0,0,2012-11-07 03:05:24.446000+00:00,2012-11-11 03:00:00.003000+0000,Tangerang,NaN,NaN,...,1,0,0,95.909877,88.922436,7.0,7.0,20,adult,589.922435
27,10329108,1940922842,0,0,0,2012-11-07 03:05:24.446000+00:00,2012-11-09 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,0,44.909876,37.922435,6.0,6.0,20,adult,589.922435
28,10329108,788687116,0,0,0,2012-11-07 03:05:24.446000+00:00,2012-11-09 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,0,44.909876,37.922435,6.0,6.0,20,adult,589.922435
29,10329108,40254649,0,1,0,2012-11-07 03:05:24.446000+00:00,2012-11-18 03:00:00.003000+0000,Tangerang,NaN,NaN,...,1,0,0,263.909877,256.922436,8.0,8.0,20,adult,589.922435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15367,4285564534,1168380708,0,0,0,2012-10-18 10:28:59.399000+00:00,2012-10-21 14:00:00.003000+0000,Bekasi,NaN,NaN,...,1,0,0,75.516834,414.651770,7.0,8.0,17,teen,16.651769
15368,4285564534,1364943007,0,0,0,2012-10-18 10:28:59.399000+00:00,2012-10-20 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,0,37.516834,376.651769,6.0,8.0,17,teen,16.651769
15369,4285564534,663767875,0,1,0,2012-10-18 10:28:59.399000+00:00,2012-10-21 06:00:00.003000+0000,Bekasi,NaN,NaN,...,1,0,0,67.516834,406.651770,6.0,8.0,17,teen,16.651769
15370,4285564534,675426904,0,0,0,2012-10-18 10:28:59.399000+00:00,2012-10-20 02:00:00.003000+0000,Yogyakarta,NaN,NaN,...,0,0,0,39.516834,378.651770,6.0,8.0,17,teen,16.651769


In [82]:
df['user_join_event_remaining_time_similarity'] = df['hrs_join_to_event']-df['avg_hrs_join_event_remaining_time']
df.head(10)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,is_same_country,is_same_state,hrs_to_event,hrs_join_to_event,hrs_to_event_category,hrs_join_to_event_category,Age,Age_Category,avg_hrs_join_event_remaining_time,user_join_event_remaining_time_similarity
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,0,0,16.115069,8.160827,5.0,4.0,22,adult,NaN,NaN
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,0,0,19.115069,11.160828,5.0,4.0,22,adult,NaN,NaN
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,0,0,573.615069,565.660828,8.0,8.0,22,adult,NaN,NaN
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,0,0,85.115069,77.160828,7.0,7.0,22,adult,NaN,NaN
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,0,0,83.115069,75.160828,7.0,7.0,22,adult,NaN,NaN
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,0,0,85.115069,77.160828,7.0,7.0,22,adult,NaN,NaN
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,118.192884,114.211470,7.0,7.0,14,teen,NaN,NaN
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,1,0,101.192044,97.211471,7.0,7.0,14,teen,NaN,NaN
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,22.179403,18.211470,5.0,5.0,14,teen,NaN,NaN
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,0,0,0.000000,0.000000,0.0,0.0,14,teen,NaN,NaN


In [83]:
print(len(df[df['user_join_event_remaining_time_similarity'].isna()==False]))
print(len(df))
print(len(train))

4154
15398
15398


In [84]:
event_attendees_yes_maybe = pd.concat([event_attendees_yes_df,event_attendees_maybe_df])
print(len(event_attendees_yes_maybe))
print(len(event_attendees_yes_df))
print(len(event_attendees_maybe_df))
print(len(event_attendees_yes_df) + len(event_attendees_maybe_df))

1357012
833121
523891
1357012


In [85]:
event_weekday_pref = event_attendees_yes_maybe.merge(train_user, how = 'inner', left_on='user_id', right_on='user')
event_weekday_pref.head(10)

,event,user_id,user
0,2370583755,338395830,338395830
1,1389928885,2500150280,2500150280
2,3753704467,3118355226,3118355226
3,235476646,2768571751,2768571751
4,606080367,3621549603,3621549603
5,444507187,2713002362,2713002362
6,444507187,4244109605,4244109605
7,401945277,1912059312,1912059312
8,3208056309,743008535,743008535
9,416330287,3725766398,3725766398


In [86]:
len(event_weekday_pref)

1438

In [87]:
event_weekday_pref.drop_duplicates()
len(event_weekday_pref)

1438

In [88]:
event_weekday_pref = event_weekday_pref.merge(events_df[['event_id','start_time']], how = 'inner', left_on = 'event', right_on = 'event_id')
event_weekday_pref.head(10)

,event,user_id,user,event_id,start_time
0,2370583755,338395830,338395830,2370583755,2012-12-14 04:00:00.003000+0000
1,1389928885,2500150280,2500150280,1389928885,2012-12-12 03:00:00.003000+0000
2,3753704467,3118355226,3118355226,3753704467,2012-12-12 12:00:00.002000+0000
3,235476646,2768571751,2768571751,235476646,2012-11-12 02:00:00.003000+0000
4,606080367,3621549603,3621549603,606080367,2012-12-14 03:00:00.003000+0000
5,444507187,2713002362,2713002362,444507187,2012-09-23 00:00:00.001000+0000
6,444507187,4244109605,4244109605,444507187,2012-09-23 00:00:00.001000+0000
7,401945277,1912059312,1912059312,401945277,2012-12-02 05:00:00.003000+0000
8,3208056309,743008535,743008535,3208056309,2012-08-31 23:00:00.003000+0000
9,416330287,3725766398,3725766398,416330287,2012-11-11 04:00:00.003000+0000


In [89]:
print(len(event_weekday_pref))
event_weekday_pref.info()

1421
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1421 entries, 0 to 1420
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   event       1421 non-null   int64 
 1   user_id     1421 non-null   Int64 
 2   user        1421 non-null   int64 
 3   event_id    1421 non-null   int64 
 4   start_time  1421 non-null   object
dtypes: Int64(1), int64(3), object(1)
memory usage: 57.0+ KB


In [90]:
event_weekday_pref['event_weekday']=pd.to_datetime(event_weekday_pref['start_time']).dt.strftime('%A')
event_weekday_pref.head(10)

,event,user_id,user,event_id,start_time,event_weekday
0,2370583755,338395830,338395830,2370583755,2012-12-14 04:00:00.003000+0000,Friday
1,1389928885,2500150280,2500150280,1389928885,2012-12-12 03:00:00.003000+0000,Wednesday
2,3753704467,3118355226,3118355226,3753704467,2012-12-12 12:00:00.002000+0000,Wednesday
3,235476646,2768571751,2768571751,235476646,2012-11-12 02:00:00.003000+0000,Monday
4,606080367,3621549603,3621549603,606080367,2012-12-14 03:00:00.003000+0000,Friday
5,444507187,2713002362,2713002362,444507187,2012-09-23 00:00:00.001000+0000,Sunday
6,444507187,4244109605,4244109605,444507187,2012-09-23 00:00:00.001000+0000,Sunday
7,401945277,1912059312,1912059312,401945277,2012-12-02 05:00:00.003000+0000,Sunday
8,3208056309,743008535,743008535,3208056309,2012-08-31 23:00:00.003000+0000,Friday
9,416330287,3725766398,3725766398,416330287,2012-11-11 04:00:00.003000+0000,Sunday


In [91]:
#event_weekday_pref[event_weekday_pref['user_id']==13301498]

In [92]:
event_weekday_pref_df = pd.DataFrame(event_weekday_pref.groupby(['user_id','event_weekday']).agg(event_count=('event_id','nunique'))).reset_index()
event_weekday_pref_df.head(10)

,user_id,event_weekday,event_count
0,13301498,Friday,1
1,13301498,Saturday,1
2,13301498,Tuesday,3
3,19283444,Monday,1
4,20041353,Sunday,1
5,24978365,Monday,1
6,28806872,Saturday,1
7,43026617,Friday,1
8,65801702,Saturday,1
9,65801702,Sunday,1


In [93]:
event_weekday_pref_pivot_df = pd.DataFrame(event_weekday_pref_df.pivot(index = 'user_id',columns = 'event_weekday',values = 'event_count')).reset_index()

In [94]:
print(len(event_weekday_pref_pivot_df))
event_weekday_pref_pivot_df.head(10)

448


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

event_weekday,user_id,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,13301498,1.0,NaN,1.0,NaN,NaN,3.0,NaN
1,19283444,NaN,1.0,NaN,NaN,NaN,NaN,NaN
2,20041353,NaN,NaN,NaN,1.0,NaN,NaN,NaN
3,24978365,NaN,1.0,NaN,NaN,NaN,NaN,NaN
4,28806872,NaN,NaN,1.0,NaN,NaN,NaN,NaN
5,43026617,1.0,NaN,NaN,NaN,NaN,NaN,NaN
6,65801702,NaN,NaN,1.0,1.0,2.0,NaN,1.0
7,69302861,NaN,NaN,NaN,NaN,NaN,1.0,NaN
8,72298333,NaN,NaN,1.0,NaN,NaN,NaN,NaN
9,81458678,3.0,NaN,1.0,1.0,NaN,NaN,NaN


In [95]:
print(len(train))
len(df)

15398


15398

In [96]:
df = df.merge(event_weekday_pref_pivot_df, how = 'left', left_on = 'user', right_on = 'user_id')
print(len(df))
print(len(train))

15398
15398


In [97]:
#df[df['user']==13301498]
df.drop(columns = ['user_id'], inplace=True)

In [98]:
event_count_df = pd.DataFrame(event_weekday_pref.groupby('user_id').agg(total_event_count = ('event_id','nunique'))).reset_index()
event_count_df.head(10)

,user_id,total_event_count
0,13301498,5
1,19283444,1
2,20041353,1
3,24978365,1
4,28806872,1
5,43026617,1
6,65801702,5
7,69302861,1
8,72298333,1
9,81458678,5


In [99]:
event_count_df[event_count_df['user_id']==13301498]

,user_id,total_event_count
0,13301498,5


In [100]:
df = df.merge(event_count_df, how = 'left', left_on = 'user', right_on ='user_id')
print(len(df))
df.head(10)

15398


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,user_join_event_remaining_time_similarity,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday,user_id,total_event_count
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN


In [101]:
df.drop(columns = ['user_id'], inplace = True)
df.head(10)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,avg_hrs_join_event_remaining_time,user_join_event_remaining_time_similarity,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday,total_event_count
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [102]:
df['pref_sunday'] = df.apply(lambda row: row['Sunday']/row['total_event_count'] if not pd.isna(row['Sunday']) else 0, axis = 1)
df['pref_monday'] = df.apply(lambda row: row['Monday']/row['total_event_count'] if not pd.isna(row['Monday']) else 0, axis = 1)
df['pref_tuesday'] = df.apply(lambda row: row['Tuesday']/row['total_event_count'] if not pd.isna(row['Tuesday']) else 0, axis = 1)
df['pref_wednesday'] = df.apply(lambda row: row['Wednesday']/row['total_event_count'] if not pd.isna(row['Wednesday']) else 0, axis = 1)
df['pref_thursday'] = df.apply(lambda row: row['Thursday']/row['total_event_count'] if not pd.isna(row['Thursday']) else 0, axis = 1)
df['pref_friday'] = df.apply(lambda row: row['Friday']/row['total_event_count'] if not pd.isna(row['Friday']) else 0, axis = 1)
df['pref_saturday'] = df.apply(lambda row: row['Saturday']/row['total_event_count'] if not pd.isna(row['Saturday']) else 0, axis = 1)
print(len(df))
df.head(10)

15398


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,Tuesday,Wednesday,total_event_count,pref_sunday,pref_monday,pref_tuesday,pref_wednesday,pref_thursday,pref_friday,pref_saturday
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [103]:
df[df['user']==13301498]

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,Tuesday,Wednesday,total_event_count,pref_sunday,pref_monday,pref_tuesday,pref_wednesday,pref_thursday,pref_friday,pref_saturday
31,13301498,3950787980,0,1,0,2012-11-07 12:30:08.950000+00:00,2012-11-08 13:00:00.003000+0000,Medan,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
32,13301498,3499428374,0,1,0,2012-11-07 12:30:04.340000+00:00,2012-11-11 03:00:00.003000+0000,NaN,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
33,13301498,3802650890,0,0,0,2012-11-07 12:30:04.340000+00:00,2012-11-10 03:00:00.003000+0000,NaN,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
34,13301498,97217712,0,0,0,2012-10-24 01:00:14.160000+00:00,2012-10-29 00:00:00.001000+0000,NaN,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
35,13301498,2401458775,0,0,0,2012-10-24 01:00:14.160000+00:00,2012-10-26 13:00:00.003000+0000,NaN,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
36,13301498,1737420203,0,0,0,2012-10-24 01:00:14.160000+00:00,2012-12-15 15:00:00.003000+0000,NaN,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
37,13301498,907302600,0,1,0,2012-10-24 01:00:14.160000+00:00,2012-10-27 13:00:00.003000+0000,Medan,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
38,13301498,457253413,0,1,0,2012-11-07 12:30:15.535000+00:00,2012-11-10 03:00:00.003000+0000,NaN,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
39,13301498,2198969023,0,0,0,2012-10-24 01:00:14.160000+00:00,2012-10-28 05:00:00.003000+0000,NaN,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2
40,13301498,955398943,0,0,0,2012-10-24 01:00:14.160000+00:00,2012-11-23 13:30:00.003000+0000,Medan,NaN,NaN,...,3.0,NaN,5.0,0.0,0.0,0.6,0.0,0.0,0.2,0.2


In [104]:
df = df.merge(events_df[['event_id','user_id']], how = 'left', left_on='event', right_on='event_id')
print(len(df))
df.drop(columns = ['event_id'],inplace = True)
df.rename(columns = {'user_id':'host_id'}, inplace=True)
len(df)

15398


15398

In [105]:
df.head(10)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,Wednesday,total_event_count,pref_sunday,pref_monday,pref_tuesday,pref_wednesday,pref_thursday,pref_friday,pref_saturday,host_id
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4106419938
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016654644
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3639934255
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97461525
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3639934255
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3286716293
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2939696577
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1618377432
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,415464198
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,937597069


In [106]:
user_friends_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38202 entries, 0 to 38201
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   user          38202 non-null  int64 
 1   friends       38063 non-null  object
 2   friends_list  38202 non-null  object
dtypes: int64(1), object(2)
memory usage: 895.5+ KB


In [107]:
user_friend_event_host_df = user_friends_df.merge(train_user, how = 'inner',left_on='user',right_on='user')
user_friend_event_host_df.drop(columns = ['friends'],inplace=True)
print(len(user_friend_event_host_df))
print(len(user_friends_df))
print(len(train_user))
user_friend_event_host_df.head(10)

2034
38202
2034


,user,friends_list
0,4101519751,"[1942428241, 2461664561, 955068058, 1149825431..."
1,2163304477,"[3369949871, 3473964934, 478577931, 1623834088..."
2,397493741,"[1028431735, 3729496862, 1759482351, 130248535..."
3,383523902,"[1602508705, 1481758665, 1712447383, 504298462..."
4,3554343061,"[2504814806, 2128063748, 3434282868, 283250427..."
5,1538685648,"[1730097234, 2297918893, 3410958855, 116533629..."
6,3251568850,"[4087658074, 2142663768, 1935004076, 559655339..."
7,2879905973,"[462549787, 3492306608, 263959637, 1182491096,..."
8,3095718598,"[3560134316, 230235488, 1653038242, 2469580915..."
9,2008761267,"[3139897898, 1660142527, 962487827, 4176864294..."


In [108]:
#l = list(user_friend_event_host_df[user_friend_event_host_df['user']==4101519751]['friends_list'])

In [109]:
def check_host_is_friend(row):
    friend_list = list(user_friend_event_host_df[user_friend_event_host_df['user']== row['user']]['friends_list'])
    if pd.isna(row['host_id']):
        return 0
    else:
        if str(row['host_id']) in friend_list[0]:
            return 1
        else:
            return 0

In [110]:
df['is_host_friend'] = df.apply(lambda row:check_host_is_friend(row), axis=1)
df.head(10)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,total_event_count,pref_sunday,pref_monday,pref_tuesday,pref_wednesday,pref_thursday,pref_friday,pref_saturday,host_id,is_host_friend
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4106419938,0
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016654644,0
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3639934255,0
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97461525,0
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3639934255,0
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3286716293,0
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2939696577,0
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1618377432,0
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,415464198,0
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,937597069,0


In [111]:
len(df[df['is_host_friend']==1])

567

In [112]:
events_desc_details_tfidf_df = events_desc_details_df.copy()
print(len(events_desc_details_df))
print(len(events_desc_details_tfidf_df))
events_desc_details_tfidf_df.columns

3137972
3137972


Index(['event_id', 'c_1', 'c_2', 'c_3', 'c_4', 'c_5', 'c_6', 'c_7', 'c_8',
       'c_9',
       ...
       'c_92', 'c_93', 'c_94', 'c_95', 'c_96', 'c_97', 'c_98', 'c_99', 'c_100',
       'c_other'],
      dtype='object', length=102)

In [113]:
events_desc_details_tfidf_df['event_desc_total_words'] = events_desc_details_tfidf_df[col_list].sum(axis = 1)
events_desc_details_tfidf_df.head(10)

,event_id,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,...,c_93,c_94,c_95,c_96,c_97,c_98,c_99,c_100,c_other,event_desc_total_words
0,684921758,2,0,2,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,9,16
1,244999119,2,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,7,17
2,3928440935,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,12,14
3,2582345152,1,0,2,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,8,16
4,1051165850,1,1,0,0,0,0,0,2,0,...,0,0,0,0,0,0,0,0,9,24
5,1212611096,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,22,22
6,3689283674,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,28,35
7,2584113432,0,0,2,0,0,33,0,3,1,...,0,0,0,0,0,0,0,0,354,413
8,3365728297,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,25,28
9,2912638473,1,0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,3,13


In [114]:
tf_col_list = []
for col in col_list:
    events_desc_details_tfidf_df[col + '_TF'] = events_desc_details_tfidf_df[col]/events_desc_details_tfidf_df['event_desc_total_words']
    tf_col_list.append(col + '_TF')
print(tf_col_list)

['c_1_TF', 'c_2_TF', 'c_3_TF', 'c_4_TF', 'c_5_TF', 'c_6_TF', 'c_7_TF', 'c_8_TF', 'c_9_TF', 'c_10_TF', 'c_11_TF', 'c_12_TF', 'c_13_TF', 'c_14_TF', 'c_15_TF', 'c_16_TF', 'c_17_TF', 'c_18_TF', 'c_19_TF', 'c_20_TF', 'c_21_TF', 'c_22_TF', 'c_23_TF', 'c_24_TF', 'c_25_TF', 'c_26_TF', 'c_27_TF', 'c_28_TF', 'c_29_TF', 'c_30_TF', 'c_31_TF', 'c_32_TF', 'c_33_TF', 'c_34_TF', 'c_35_TF', 'c_36_TF', 'c_37_TF', 'c_38_TF', 'c_39_TF', 'c_40_TF', 'c_41_TF', 'c_42_TF', 'c_43_TF', 'c_44_TF', 'c_45_TF', 'c_46_TF', 'c_47_TF', 'c_48_TF', 'c_49_TF', 'c_50_TF', 'c_51_TF', 'c_52_TF', 'c_53_TF', 'c_54_TF', 'c_55_TF', 'c_56_TF', 'c_57_TF', 'c_58_TF', 'c_59_TF', 'c_60_TF', 'c_61_TF', 'c_62_TF', 'c_63_TF', 'c_64_TF', 'c_65_TF', 'c_66_TF', 'c_67_TF', 'c_68_TF', 'c_69_TF', 'c_70_TF', 'c_71_TF', 'c_72_TF', 'c_73_TF', 'c_74_TF', 'c_75_TF', 'c_76_TF', 'c_77_TF', 'c_78_TF', 'c_79_TF', 'c_80_TF', 'c_81_TF', 'c_82_TF', 'c_83_TF', 'c_84_TF', 'c_85_TF', 'c_86_TF', 'c_87_TF', 'c_88_TF', 'c_89_TF', 'c_90_TF', 'c_91_TF', 'c_92_T

/tmp/ipykernel_35/2841804000.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  events_desc_details_tfidf_df[col + '_TF'] = events_desc_details_tfidf_df[col]/events_desc_details_tfidf_df['event_desc_total_words']
/tmp/ipykernel_35/2841804000.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  events_desc_details_tfidf_df[col + '_TF'] = events_desc_details_tfidf_df[col]/events_desc_details_tfidf_df['event_desc_total_words']
/tmp/ipykernel_35/2841804000.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually th

In [115]:
print(events_desc_details_tfidf_df.columns)
events_desc_details_tfidf_df.head(10)

Index(['event_id', 'c_1', 'c_2', 'c_3', 'c_4', 'c_5', 'c_6', 'c_7', 'c_8',
       'c_9',
       ...
       'c_92_TF', 'c_93_TF', 'c_94_TF', 'c_95_TF', 'c_96_TF', 'c_97_TF',
       'c_98_TF', 'c_99_TF', 'c_100_TF', 'c_other_TF'],
      dtype='object', length=204)


,event_id,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,...,c_92_TF,c_93_TF,c_94_TF,c_95_TF,c_96_TF,c_97_TF,c_98_TF,c_99_TF,c_100_TF,c_other_TF
0,684921758,2,0,2,0,0,0,0,0,0,...,0.000000,0.0625,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.562500
1,244999119,2,0,2,0,0,0,0,0,0,...,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.411765
2,3928440935,0,0,0,0,0,0,0,0,0,...,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.857143
3,2582345152,1,0,2,1,0,0,0,0,0,...,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.500000
4,1051165850,1,1,0,0,0,0,0,2,0,...,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.375000
5,1212611096,0,0,0,0,0,0,0,0,0,...,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1.000000
6,3689283674,0,0,0,1,0,0,0,0,0,...,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.800000
7,2584113432,0,0,2,0,0,33,0,3,1,...,0.004843,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.857143
8,3365728297,0,0,0,0,0,0,0,0,0,...,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.035714,0.0,0.892857
9,2912638473,1,0,1,0,1,0,0,0,0,...,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.230769


In [116]:
#event_attendees_df.info()

In [117]:
event_yes_maybe_desc_df = event_attendees_yes_maybe.merge(train_user, how='inner', left_on = 'user_id', right_on = 'user')
print(len(event_yes_maybe_desc_df))
event_yes_maybe_desc_df.head(10)

1438


,event,user_id,user
0,2370583755,338395830,338395830
1,1389928885,2500150280,2500150280
2,3753704467,3118355226,3118355226
3,235476646,2768571751,2768571751
4,606080367,3621549603,3621549603
5,444507187,2713002362,2713002362
6,444507187,4244109605,4244109605
7,401945277,1912059312,1912059312
8,3208056309,743008535,743008535
9,416330287,3725766398,3725766398


In [118]:
event_yes_maybe_desc_df.drop(columns='user_id', inplace =True)

In [119]:
event_tf_col_list = ['event_id'] + tf_col_list
print(event_tf_col_list)

['event_id', 'c_1_TF', 'c_2_TF', 'c_3_TF', 'c_4_TF', 'c_5_TF', 'c_6_TF', 'c_7_TF', 'c_8_TF', 'c_9_TF', 'c_10_TF', 'c_11_TF', 'c_12_TF', 'c_13_TF', 'c_14_TF', 'c_15_TF', 'c_16_TF', 'c_17_TF', 'c_18_TF', 'c_19_TF', 'c_20_TF', 'c_21_TF', 'c_22_TF', 'c_23_TF', 'c_24_TF', 'c_25_TF', 'c_26_TF', 'c_27_TF', 'c_28_TF', 'c_29_TF', 'c_30_TF', 'c_31_TF', 'c_32_TF', 'c_33_TF', 'c_34_TF', 'c_35_TF', 'c_36_TF', 'c_37_TF', 'c_38_TF', 'c_39_TF', 'c_40_TF', 'c_41_TF', 'c_42_TF', 'c_43_TF', 'c_44_TF', 'c_45_TF', 'c_46_TF', 'c_47_TF', 'c_48_TF', 'c_49_TF', 'c_50_TF', 'c_51_TF', 'c_52_TF', 'c_53_TF', 'c_54_TF', 'c_55_TF', 'c_56_TF', 'c_57_TF', 'c_58_TF', 'c_59_TF', 'c_60_TF', 'c_61_TF', 'c_62_TF', 'c_63_TF', 'c_64_TF', 'c_65_TF', 'c_66_TF', 'c_67_TF', 'c_68_TF', 'c_69_TF', 'c_70_TF', 'c_71_TF', 'c_72_TF', 'c_73_TF', 'c_74_TF', 'c_75_TF', 'c_76_TF', 'c_77_TF', 'c_78_TF', 'c_79_TF', 'c_80_TF', 'c_81_TF', 'c_82_TF', 'c_83_TF', 'c_84_TF', 'c_85_TF', 'c_86_TF', 'c_87_TF', 'c_88_TF', 'c_89_TF', 'c_90_TF', 'c_91_

In [120]:
event_yes_maybe_desc_df.columns

Index(['event', 'user'], dtype='object')

In [121]:
event_yes_maybe_desc_df = event_yes_maybe_desc_df.merge(events_desc_details_df, how = 'left', left_on = 'event', right_on='event_id')
event_yes_maybe_desc_df = event_yes_maybe_desc_df.merge(events_desc_details_tfidf_df[event_tf_col_list], how = 'left', left_on = 'event', right_on='event_id')
print(len(event_yes_maybe_desc_df))
event_yes_maybe_desc_df.head(10)

1438


,event,user,event_id_x,c_1,c_2,c_3,c_4,c_5,c_6,c_7,...,c_92_TF,c_93_TF,c_94_TF,c_95_TF,c_96_TF,c_97_TF,c_98_TF,c_99_TF,c_100_TF,c_other_TF
0,2370583755,338395830,2.370584e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.800000
1,1389928885,2500150280,1.389929e+09,0.0,2.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.560000
2,3753704467,3118355226,3.753704e+09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.000000
3,235476646,2768571751,2.354766e+08,26.0,13.0,14.0,6.0,17.0,5.0,6.0,...,0.0,0.002342,0.000000,0.002342,0.002342,0.007026,0.002342,0.0,0.002342,0.519906
4,606080367,3621549603,6.060804e+08,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.000000,0.021277,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.680851
5,444507187,2713002362,4.445072e+08,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.645161
6,444507187,4244109605,4.445072e+08,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.645161
7,401945277,1912059312,4.019453e+08,4.0,0.0,1.0,0.0,4.0,10.0,2.0,...,0.0,0.004149,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.004149,0.825726
8,3208056309,743008535,3.208056e+09,4.0,1.0,6.0,1.0,1.0,1.0,0.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.469880
9,416330287,3725766398,4.163303e+08,1.0,0.0,0.0,0.0,1.0,11.0,2.0,...,0.0,0.000000,0.000000,0.007812,0.000000,0.000000,0.000000,0.0,0.007812,0.804688


In [122]:
event_yes_maybe_desc_df.drop(columns = 'event_id_x', inplace = True)

In [123]:
event_yes_maybe_desc_df = event_yes_maybe_desc_df.fillna(0)

In [124]:
print(event_yes_maybe_desc_df[['event','user']].nunique())
len(event_yes_maybe_desc_df)

event    1031
user      448
dtype: int64


1438

In [125]:
tf_col_event =['event','user'] + tf_col_list
len(tf_col_event)

103

In [126]:
_event_lookup = None
_user_event_lookup = None
_tf_cols = None
_user_tf_cols = None

In [127]:
#events_desc_details_tfidf_df.head(10)

In [128]:
def initialize_lookups(events_desc_details_tfidf_df,event_yes_maybe_desc_df,event_tf_col_list,tf_col_event):
    global _event_lookup, _user_event_lookup, _tf_cols, _user_tf_cols
    #_tf_cols = [col for col in event_tf_col_list if col not in ['event_id','c_other_TF']]
    #_user_tf_cols = [col for col in tf_col_event if col not in ['event','c_other_TF','user']]
    _tf_cols = [col for col in event_tf_col_list if col not in ['event_id']]
    _user_tf_cols = [col for col in tf_col_event if col not in ['event','user']]
    _event_lookup = {}
    _user_event_lookup = {}
    for _,row in events_desc_details_tfidf_df.iterrows():
        event_id = row['event_id']
        features = row[_tf_cols].values
        _event_lookup[event_id] = features
    for user, group in event_yes_maybe_desc_df.groupby('user'):
        events = group['event'].values
        features = group[_user_tf_cols].values
        _user_event_lookup[user] = {
            'event_id':events,
            'features':features
        }

In [129]:
def get_event_tf_matrix_optimized(row,event_tf_col_list,tf_col_event,similarity_threshold=0.7):
    cos_sim_user_event = {}
    try:
        if row['event'] not in _event_lookup:
            return cos_sim_user_event
        event_features = _event_lookup[row['event']].reshape(1,-1)
        if row['user'] not in _user_event_lookup:
            return cos_sim_user_event
        user_data = _user_event_lookup[row['user']]
        user_event = user_data['event_id']
        user_features = user_data['features']
        if len(user_features)==0:
            return cos_sim_user_event
        cos_sim_matrix = cosine_similarity(event_features,user_features)[0]
        #print(cos_sim_matrix)
        high_sim_mask = cos_sim_matrix>=similarity_threshold
        if np.any(high_sim_mask):
            high_sim_events = user_event[high_sim_mask]
            #print(high_sim_events)
            high_sim_score = cos_sim_matrix[high_sim_mask]
            #print(high_sim_score)
            cos_sim_user_event = dict(zip(high_sim_events,np.round(high_sim_score,4)))
        #print(cos_sim_user_event)
        return cos_sim_user_event
    except Exception as e:
        print(f"Error processing row with event {row.get('event', 'unknown')}, user {row.get('user', 'unknown')}: {e}")
        return cos_sim_user_event

In [130]:
initialize_lookups(events_desc_details_tfidf_df,event_yes_maybe_desc_df,event_tf_col_list,tf_col_event)
print("Total records in _event_lookup: ", len(_event_lookup))
print("Total records in _user_event_lookup:", len(_user_event_lookup))

Total records in _event_lookup:  3137972
Total records in _user_event_lookup: 448


In [131]:
df.loc[:, 'cosine_similarity'] = df.apply(lambda row:get_event_tf_matrix_optimized(row,event_tf_col_list,tf_col_event,0.7),axis=1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,pref_sunday,pref_monday,pref_tuesday,pref_wednesday,pref_thursday,pref_friday,pref_saturday,host_id,is_host_friend,cosine_similarity
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4106419938,0,{}
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2016654644,0,{}
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3639934255,0,{}
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97461525,0,{}
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3639934255,0,{}
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3286716293,0,{}
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2939696577,0,{}
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1618377432,0,{}
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,415464198,0,{}
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,937597069,0,{}


In [132]:
df[df['cosine_similarity'].astype(str) != '{}']

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,pref_sunday,pref_monday,pref_tuesday,pref_wednesday,pref_thursday,pref_friday,pref_saturday,host_id,is_host_friend,cosine_similarity
31,13301498,3950787980,0,1,0,2012-11-07 12:30:08.950000+00:00,2012-11-08 13:00:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,0.6,0.0,0.0,0.2,0.2,3604594799,0,"{2952943730: 0.9978, 3245843888: 0.9971, 36322..."
32,13301498,3499428374,0,1,0,2012-11-07 12:30:04.340000+00:00,2012-11-11 03:00:00.003000+0000,NaN,NaN,NaN,...,0.0,0.0,0.6,0.0,0.0,0.2,0.2,3007418148,0,"{2952943730: 0.9956, 3245843888: 0.9948, 36322..."
33,13301498,3802650890,0,0,0,2012-11-07 12:30:04.340000+00:00,2012-11-10 03:00:00.003000+0000,NaN,NaN,NaN,...,0.0,0.0,0.6,0.0,0.0,0.2,0.2,2935562301,0,"{2952943730: 0.9917, 3245843888: 0.991, 363226..."
34,13301498,97217712,0,0,0,2012-10-24 01:00:14.160000+00:00,2012-10-29 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,0.6,0.0,0.0,0.2,0.2,1063055103,0,"{2952943730: 0.9386, 3245843888: 0.9379, 36322..."
35,13301498,2401458775,0,0,0,2012-10-24 01:00:14.160000+00:00,2012-10-26 13:00:00.003000+0000,NaN,NaN,NaN,...,0.0,0.0,0.6,0.0,0.0,0.2,0.2,1266542266,0,"{2952943730: 0.9963, 3245843888: 0.9956, 36322..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15367,4285564534,1168380708,0,0,0,2012-10-18 10:28:59.399000+00:00,2012-10-21 14:00:00.003000+0000,Bekasi,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2774082477,0,{1697930615: 0.9487}
15368,4285564534,1364943007,0,0,0,2012-10-18 10:28:59.399000+00:00,2012-10-20 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1529839966,0,{1697930615: 0.933}
15369,4285564534,663767875,0,1,0,2012-10-18 10:28:59.399000+00:00,2012-10-21 06:00:00.003000+0000,Bekasi,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1931464264,1,{1697930615: 0.9466}
15370,4285564534,675426904,0,0,0,2012-10-18 10:28:59.399000+00:00,2012-10-20 02:00:00.003000+0000,Yogyakarta,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,910400279,0,{1697930615: 0.9339}


In [133]:
print(len(event_attendees_yes_maybe))
event_train_user_attendees_yes_maybe = event_attendees_yes_maybe.merge(train_user, how = 'inner', left_on='user_id', right_on='user')
event_train_user_attendees_yes_maybe.drop(columns = 'user',inplace = True)

1357012


In [134]:
event_host_df = events_full_df[['event_id','user_id']]
print(len(event_host_df))
event_host_df.nunique()

3137972


event_id    3137972
user_id     1918835
dtype: int64

In [135]:
_event_host_lookup = None
_user_event_lookup_yes_maybe = None
_event_host_full_data = None

In [136]:
def intialize_lookups_event_host_user_response(event_host_df,event_train_user_attendees_yes_maybe,events_full_df):
    global _event_host_lookup,_user_event_lookup_yes_maybe,_event_host_full_data
    _event_host_lookup = {}
    _user_event_lookup_yes_maybe ={}
    _event_host_full_data = {}
    for host,group in event_host_df.groupby('user_id'):
        _event_host_lookup[host]=group['event_id'].values
    for user, group in event_train_user_attendees_yes_maybe.groupby('user_id'):
        _user_event_lookup_yes_maybe[user]=group['event'].values
    for _,row in events_full_df.iterrows():
        _event_host_full_data[row['event_id']]=row['user_id']

In [137]:
'''intialize_lookups_event_host_user_response(event_host_df,event_train_user_attendees_yes_maybe,events_full_df)
print(len(_event_host_lookup))
print(len(_user_event_lookup_yes_maybe))
print(len(_event_host_full_data))'''
#_event_host_lookup

'intialize_lookups_event_host_user_response(event_host_df,event_train_user_attendees_yes_maybe,events_full_df)\nprint(len(_event_host_lookup))\nprint(len(_user_event_lookup_yes_maybe))\nprint(len(_event_host_full_data))'

In [138]:
#event_train_user_attendees_yes_maybe[event_train_user_attendees_yes_maybe['user_id']==85640691]

In [139]:
def get_prev_host_events_attended_yes_maybe(row):
    prev_host_events_attended = []
    try:
        host_id = _event_host_full_data[row['event']]
        if host_id not in _event_host_lookup:
            return len(prev_host_events_attended)
        host_events = _event_host_lookup[host_id]
        if row['user'] not in _user_event_lookup_yes_maybe:
            return len(prev_host_events_attended)
        events_attended = _user_event_lookup_yes_maybe[row['user']]
        prev_host_events_attended = list(set(host_events).intersection(set(events_attended)))
        #if len(prev_host_events_attended)>0:
            #print(prev_host_events_attended)
            #print("Event:", row['event'], "  User:", row['user'])
        return len(prev_host_events_attended)
    except Exception as e:
        print(f"Error processing row with event {row.get('event', 'unknown')}, user {row.get('user', 'unknown')}: {e}")
        return len(prev_host_events_attended)

In [140]:
'''df.loc[:,'host_event_prev_attended'] = df.apply(lambda row:get_prev_host_events_attended_yes_maybe(row),axis=1)
df.head(10)'''

"df.loc[:,'host_event_prev_attended'] = df.apply(lambda row:get_prev_host_events_attended_yes_maybe(row),axis=1)\ndf.head(10)"

In [141]:
_event_user_train_lookup = None
_user_friend_lookup = None

In [142]:
def initialize_event_user_friend_lookups(train, user_friends_df):
    global _event_user_train_lookup, _user_friend_lookup
    _event_user_train_lookup = {}
    _user_friend_lookup = {}
    friend_list =[]
    try:
        for event, group in df[['event','user']].groupby('event'):
            _event_user_train_lookup[event] = group['user'].values
        for _, row in user_friends_df.iterrows():
            #_user_friend_lookup[row['user']] = row['friends_list']
            friend_list = row['friends_list']
            if len(friend_list)>0 and isinstance(friend_list[0],str):
                friend_list = [int(x) for x in friend_list]
            _user_friend_lookup[row['user']] = friend_list
    except Exception as e:
        print("Exception occured for user:", row['user'],". Error:", {e})

In [143]:
initialize_event_user_friend_lookups(train, user_friends_df)
print(len(_event_user_train_lookup))
print(len(_user_friend_lookup))

8846
38202


In [144]:
def get_user_friends_event_to_attend_ratio(row):
    friends_registered_event = []
    try:
        if row['event'] not in _event_user_train_lookup:
            return 0
        user_list = _event_user_train_lookup[row['event']]
        if row['user'] not in _user_friend_lookup:
            return 0
        user_friend_list = _user_friend_lookup[row['user']]
        friends_registered_event = list(set(user_list).intersection(set(user_friend_list)))
        user_friends_event_to_attend_ratio = np.round(len(friends_registered_event)/len(user_list),4)
        return user_friends_event_to_attend_ratio
    except Exception as e:
        print(f"Error processing row with event {row.get('event', 'unknown')}, user {row.get('user', 'unknown')}: {e}")
        return 0

In [145]:
df['friend_registered_ratio'] = df.apply(lambda row:get_user_friends_event_to_attend_ratio(row), axis = 1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,pref_monday,pref_tuesday,pref_wednesday,pref_thursday,pref_friday,pref_saturday,host_id,is_host_friend,cosine_similarity,friend_registered_ratio
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,4106419938,0,{},0.0000
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,2016654644,0,{},0.0000
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,3639934255,0,{},0.0000
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,97461525,0,{},0.0000
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,3639934255,0,{},0.0000
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,3286716293,0,{},0.0000
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,2939696577,0,{},0.0909
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,1618377432,0,{},0.0323
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,415464198,0,{},0.0312
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,937597069,0,{},0.0000


In [146]:
print(len(df[df['friend_registered_ratio']>0.3]))
len(df[(df['friend_registered_ratio']<0.5) & (df['friend_registered_ratio']>0)])

276


1317

In [147]:
_event_attendees_response_lookup=None

In [148]:
def event_attendees_response_lookup(event_attendees_df):
    global _event_attendees_response_lookup
    _event_attendees_response_lookup={}
    try:
        for _,row in event_attendees_df.iterrows():
            response_data = {}
            if not pd.isna(row['invited']):
                response_data['invited'] = list(map(int,row['invited'].split()))
            else:
                response_data['invited'] = []
            if not pd.isna(row['maybe']):
                response_data['maybe'] = list(map(int,row['maybe'].split()))
            else:
                response_data['maybe']=[]
            if not pd.isna(row['yes']):
                response_data['yes'] = list(map(int, row['yes'].split()))
            else:
                response_data['yes'] = []
            if not pd.isna(row['no']):
                response_data['no'] = list(map(int, row['no'].split()))
            else:
                response_data['no'] = []
            _event_attendees_response_lookup[row['event']]=response_data
        print(len(_event_attendees_response_lookup))
        return _event_attendees_response_lookup
    except Exception as e:
        print(f"Error has occured while processing lookup. Error Details: {e}")

In [149]:
event_attendees_response_lookup(event_attendees_df)
len(_event_attendees_response_lookup)

24144


24144

In [150]:
def get_users_response_ratio(row,user_response):
    user_response_count = 0
    total_count = 0
    user_response_ratio = 0
    try:
        if row['event'] in _event_attendees_response_lookup:
            response_data = _event_attendees_response_lookup[row['event']]
            #print(response_data)
            total_count = total_count + len(response_data['invited'] + response_data['maybe'] + response_data['yes'] + response_data['no'])
            if total_count>0:
                user_response_ratio = len(response_data[user_response])/total_count
        return user_response_ratio
    except Exception as e:
        print(f"Error occured while processing event {row.get('event','unknown')} for response type {user_response}. Error details: {e}")
        return user_response_ratio

In [151]:
df.loc[:,'invited_users_ratio'] = df.apply(lambda row:get_users_response_ratio(row,'invited'),axis = 1)
df.loc[:,'attending_users_ratio'] = df.apply(lambda row:get_users_response_ratio(row,'yes'),axis = 1)
df.loc[:,'not_interested_users_ratio'] = df.apply(lambda row:get_users_response_ratio(row,'no'),axis = 1)
df.loc[:,'maybe_interested_users_ratio'] = df.apply(lambda row:get_users_response_ratio(row,'maybe'),axis = 1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,pref_friday,pref_saturday,host_id,is_host_friend,cosine_similarity,friend_registered_ratio,invited_users_ratio,attending_users_ratio,not_interested_users_ratio,maybe_interested_users_ratio
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,0.0,0.0,4106419938,0,{},0.0000,0.396552,0.137931,0.431034,0.034483
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,0.0,0.0,2016654644,0,{},0.0000,0.871429,0.071429,0.014286,0.042857
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,3639934255,0,{},0.0000,0.884288,0.048769,0.031516,0.035427
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,0.0,0.0,97461525,0,{},0.0000,0.391304,0.347826,0.000000,0.260870
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,3639934255,0,{},0.0000,0.949465,0.020662,0.013692,0.016181
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,0.0,0.0,3286716293,0,{},0.0000,0.936614,0.034303,0.008949,0.020134
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,2939696577,0,{},0.0909,0.925125,0.028286,0.016639,0.029950
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,0.0,0.0,1618377432,0,{},0.0323,0.713630,0.107357,0.075991,0.103022
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,415464198,0,{},0.0312,0.325192,0.331897,0.079023,0.263889
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0,0.0,937597069,0,{},0.0000,0.930370,0.022222,0.031111,0.016296


In [152]:
def get_user_friend_response_ratio(row,user_friends_response):
    user_friend_response_ratio = 0
    total_count = 0
    try:
        if row['user'] in _user_friend_lookup:
            friend_list = _user_friend_lookup[row['user']]
            total_count = len(friend_list)
            if row['event'] in _event_attendees_response_lookup:
                response_data = _event_attendees_response_lookup[row['event']]
                user_friend_count = len(list(set(response_data[user_friends_response]).intersection(set(friend_list))))
                if total_count > 0:
                    user_friend_response_ratio = user_friend_count/total_count
        return user_friend_response_ratio
    except Exception as e:
        print(f"Error occured while processing event {row.get('event','unknown')} for response type {user_response}. Error details: {e}")
        return user_response_ratio

In [153]:
df.loc[:,'invited_user_friend_ratio'] = df.apply(lambda row:get_user_friend_response_ratio(row,'invited'), axis=1)
df.loc[:,'attending_user_friend_ratio'] = df.apply(lambda row:get_user_friend_response_ratio(row,'yes'), axis=1)
df.loc[:,'not_interested_user_friend_ratio'] = df.apply(lambda row:get_user_friend_response_ratio(row,'no'), axis=1)
df.loc[:,'maybe_interested_user_friend_ratio'] = df.apply(lambda row:get_user_friend_response_ratio(row,'maybe'), axis=1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,cosine_similarity,friend_registered_ratio,invited_users_ratio,attending_users_ratio,not_interested_users_ratio,maybe_interested_users_ratio,invited_user_friend_ratio,attending_user_friend_ratio,not_interested_user_friend_ratio,maybe_interested_user_friend_ratio
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,{},0.0000,0.396552,0.137931,0.431034,0.034483,0.003480,0.000000,0.003480,0.000000
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,{},0.0000,0.871429,0.071429,0.014286,0.042857,0.001160,0.000000,0.000000,0.000000
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,{},0.0000,0.884288,0.048769,0.031516,0.035427,0.002320,0.000000,0.000000,0.000000
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,{},0.0000,0.391304,0.347826,0.000000,0.260870,0.000000,0.001160,0.000000,0.000000
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,{},0.0000,0.949465,0.020662,0.013692,0.016181,0.002320,0.000000,0.000000,0.000000
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,{},0.0000,0.936614,0.034303,0.008949,0.020134,0.002320,0.000000,0.000000,0.000000
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,{},0.0909,0.925125,0.028286,0.016639,0.029950,0.050336,0.000000,0.001678,0.001678
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,{},0.0323,0.713630,0.107357,0.075991,0.103022,0.060403,0.008389,0.001678,0.010067
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,{},0.0312,0.325192,0.331897,0.079023,0.263889,0.025168,0.013423,0.001678,0.015101
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,{},0.0000,0.930370,0.022222,0.031111,0.016296,0.028523,0.000000,0.000000,0.000000


In [154]:
df.loc[:,'no_of_similar_events'] = df.apply(lambda row:len(row['cosine_similarity']) if len(row['cosine_similarity'])>0 else 0,axis = 1)
df.head(10)

,user,event,invited,interested,not_interested,timestamp,event_start_time,event_city,event_state,event_zip,...,friend_registered_ratio,invited_users_ratio,attending_users_ratio,not_interested_users_ratio,maybe_interested_users_ratio,invited_user_friend_ratio,attending_user_friend_ratio,not_interested_user_friend_ratio,maybe_interested_user_friend_ratio,no_of_similar_events
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 08:00:00.002000+0000,NaN,NaN,NaN,...,0.0000,0.396552,0.137931,0.431034,0.034483,0.003480,0.000000,0.003480,0.000000,0
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-03 11:00:00.003000+0000,Yogyakarta,NaN,NaN,...,0.0000,0.871429,0.071429,0.014286,0.042857,0.001160,0.000000,0.000000,0.000000,0
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,2012-10-26 13:30:00.003000+0000,Medan,NaN,NaN,...,0.0000,0.884288,0.048769,0.031516,0.035427,0.002320,0.000000,0.000000,0.000000,0
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,NaN,NaN,NaN,...,0.0000,0.391304,0.347826,0.000000,0.260870,0.000000,0.001160,0.000000,0.000000,0
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 03:00:00.003000+0000,Medan,NaN,NaN,...,0.0000,0.949465,0.020662,0.013692,0.016181,0.002320,0.000000,0.000000,0.000000,0
5,3044012,1532377761,0,0,0,2012-10-02 15:53:05.754000+00:00,2012-10-06 05:00:00.003000+0000,Medan,NaN,NaN,...,0.0000,0.936614,0.034303,0.008949,0.020134,0.002320,0.000000,0.000000,0.000000,0
6,4236494,2352676247,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-11-04 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0909,0.925125,0.028286,0.016639,0.029950,0.050336,0.000000,0.001678,0.001678,0
7,4236494,152418051,0,1,0,2012-10-30 01:48:28.645000+00:00,2012-11-03 07:00:00.003000+0000,Port Louis Town,NaN,NaN,...,0.0323,0.713630,0.107357,0.075991,0.103022,0.060403,0.008389,0.001678,0.010067,0
8,4236494,4203627753,0,1,0,2012-10-30 01:49:14.152000+00:00,2012-10-31 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0312,0.325192,0.331897,0.079023,0.263889,0.025168,0.013423,0.001678,0.015101,0
9,4236494,110357109,0,0,0,2012-10-30 01:48:25.617000+00:00,2012-10-30 00:00:00.001000+0000,NaN,NaN,NaN,...,0.0000,0.930370,0.022222,0.031111,0.016296,0.028523,0.000000,0.000000,0.000000,0


In [155]:
train_df = df.copy()

In [156]:
print(df.shape)
train_df.shape

(15398, 55)


(15398, 55)

In [157]:
train_df.drop(columns = ['event_start_time','event_city','event_state','event_zip','event_country','birthyear','gender','location','joinedAt','loc_match_col',
                         'Age_Category','avg_hrs_join_event_remaining_time','Friday','Monday','Saturday','Sunday','Thursday','Tuesday','Wednesday',
                         'total_event_count','host_id','friend_registered_ratio','user_join_event_remaining_time_similarity',
                         'cosine_similarity','hrs_to_event_category','hrs_join_to_event_category'], inplace=True)
train_df.shape #'host_event_prev_attended'

(15398, 29)

In [158]:
train_num = list(train_df.select_dtypes(include=['int64','float64']).columns)
train_num.remove('user')
train_num.remove('event')
train_num


['invited',
 'interested',
 'not_interested',
 'is_same_city',
 'is_same_country',
 'is_same_state',
 'hrs_to_event',
 'hrs_join_to_event',
 'Age',
 'pref_sunday',
 'pref_monday',
 'pref_tuesday',
 'pref_wednesday',
 'pref_thursday',
 'pref_friday',
 'pref_saturday',
 'is_host_friend',
 'invited_users_ratio',
 'attending_users_ratio',
 'not_interested_users_ratio',
 'maybe_interested_users_ratio',
 'invited_user_friend_ratio',
 'attending_user_friend_ratio',
 'not_interested_user_friend_ratio',
 'maybe_interested_user_friend_ratio',
 'no_of_similar_events']

In [159]:
train_obj = list(train_df.select_dtypes(include='object').columns)
train_obj

['timestamp']

In [ ]:
fig, axes = plt.subplots(7,4, figsize=(20,15))
for i in range(len(train_num)):
    ax = axes[i//4,i%4]
    sns.histplot(train_df[train_num[i]], kde=True, bins=30, ax=ax)
    ax.set_title(f'Distribution of {train_num[i]}')
    ax.set_xlabel(train_num[i])
    ax.set_ylabel('Frequency')
plt.tight_layout()    
plt.show
    

/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  with pd.option_context('mode.use_inf_as_na', True):
/usr/local/lib/python3.11/dist-packages/seaborn/_oldcore.py:1119: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  

<function matplotlib.pyplot.show(close=None, block=None)>

In [184]:
def find_outliers_IQR(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    outliers = series[(series < (q1 - 1.5 * iqr)) | (series > (q3 + 1.5 * iqr))]
    return outliers

In [185]:
def handle_outliers_clip(train_df, multiplier=1.5):
    for col in ['hrs_to_event','hrs_join_to_event']:
        Q1 = train_df[col].quantile(0.25)
        Q3 = train_df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - (multiplier * IQR)
        upper_bound = Q3 + (multiplier * IQR)
        print(f"For Column {col} :- Lower: {lower_bound}, Upper: {upper_bound}")
        #X_capped[col] = np.clip(X_capped[col], lower_bound, upper_bound)

In [186]:
def winsorize_df(train_df, columns, limits=(0.05, 0.05)):
    df_winsor = train_df.copy()
    for col in columns:
        df_winsor[col] = winsorize(train_df[col], limits=limits)
    return df_winsor

In [187]:
df_winsor_num = winsorize_df(train_df,['Age','no_of_similar_events','hrs_to_event','hrs_join_to_event'])
df_winsor_num

,user,event,invited,interested,not_interested,timestamp,is_same_city,is_same_country,is_same_state,hrs_to_event,...,is_host_friend,invited_users_ratio,attending_users_ratio,not_interested_users_ratio,maybe_interested_users_ratio,invited_user_friend_ratio,attending_user_friend_ratio,not_interested_user_friend_ratio,maybe_interested_user_friend_ratio,no_of_similar_events
0,3044012,1918771225,0,0,0,2012-10-02 15:53:05.754000+00:00,0,0,0,16.115069,...,0,0.396552,0.137931,0.431034,0.034483,0.003480,0.000000,0.003480,0.000000,0
1,3044012,1502284248,0,0,0,2012-10-02 15:53:05.754000+00:00,0,0,0,19.115069,...,0,0.871429,0.071429,0.014286,0.042857,0.001160,0.000000,0.000000,0.000000,0
2,3044012,2529072432,0,1,0,2012-10-02 15:53:05.754000+00:00,0,0,0,573.615069,...,0,0.884288,0.048769,0.031516,0.035427,0.002320,0.000000,0.000000,0.000000,0
3,3044012,3072478280,0,0,0,2012-10-02 15:53:05.754000+00:00,0,0,0,85.115069,...,0,0.391304,0.347826,0.000000,0.260870,0.000000,0.001160,0.000000,0.000000,0
4,3044012,1390707377,0,0,0,2012-10-02 15:53:05.754000+00:00,0,0,0,83.115069,...,0,0.949465,0.020662,0.013692,0.016181,0.002320,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15393,4293103086,2750873665,0,0,0,2012-12-08 03:59:43.169000+00:00,0,0,0,8.004676,...,0,0.686331,0.159712,0.037410,0.116547,0.000000,0.000000,0.000000,0.002083,0
15394,4293103086,4084655790,0,0,0,2012-12-08 03:59:43.169000+00:00,0,0,0,21.004676,...,0,0.804627,0.095116,0.005141,0.095116,0.002083,0.002083,0.000000,0.000000,0
15395,4293103086,598708806,0,0,0,2012-12-08 03:59:43.169000+00:00,0,0,0,23.004676,...,0,0.727273,0.189133,0.014629,0.068966,0.006250,0.002083,0.002083,0.002083,0
15396,4293103086,604179853,0,0,0,2012-12-08 03:59:43.169000+00:00,0,0,0,104.004676,...,0,0.298900,0.154775,0.475683,0.070643,0.000000,0.000000,0.002083,0.002083,0


In [ ]:
#handle_outliers_clip(train_df,1.5)

In [188]:
corr_matrix = train_df[train_num].corr()
plt.figure(figsize=(20,20))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

In [189]:
print("Train df Age:", train_df['Age'].min())
print("Train df Age:", train_df['Age'].max())
print("Winsor Age:", df_winsor_num['Age'].min())
print("Winsor Age:", df_winsor_num['Age'].max())
print("Train no_of_similar_events:",train_df['no_of_similar_events'].min())
print("Train no_of_similar_events:",train_df['no_of_similar_events'].max())
print("Winsor no_of_similar_events:",df_winsor_num['no_of_similar_events'].min())
print("Winsor no_of_similar_events:",df_winsor_num['no_of_similar_events'].max())
print("Train hrs_to_event:",train_df['hrs_to_event'].min())
print("Train hrs_to_event:",train_df['hrs_to_event'].max())
print("Winsor hrs_to_event:",df_winsor_num['hrs_to_event'].min())
print("Winsor hrs_to_event:", df_winsor_num['hrs_to_event'].max())
print("Train hrs_join_to_event:", train_df['hrs_join_to_event'].min())
print("Train hrs_join_to_event:",train_df['hrs_join_to_event'].max())
print("Winsor hrs_join_to_event:",df_winsor_num['hrs_join_to_event'].min())
print("Winsor hrs_join_to_event:",df_winsor_num['hrs_join_to_event'].max())

Train df Age: 13
Train df Age: 2012
Winsor Age: 15
Winsor Age: 42
Train no_of_similar_events: 0
Train no_of_similar_events: 51
Winsor no_of_similar_events: 0
Winsor no_of_similar_events: 5
Train hrs_to_event: 0.0
Train hrs_to_event: 18019.939005
Winsor hrs_to_event: 0.0
Winsor hrs_to_event: 959.9774958333333
Train hrs_join_to_event: 0.0
Train hrs_join_to_event: 18028.218419166667
Winsor hrs_join_to_event: 0.0
Winsor hrs_join_to_event: 1897.9791044444444


In [226]:
y_interested = train_df[['interested']]
y_not_interested = train_df[['not_interested']]
X = df_winsor_num.copy()
X.drop(columns = ['timestamp','user','event','interested','not_interested'],inplace = True)
print(y_interested.shape)
print(y_not_interested.shape)
print(df_winsor_num.shape)
print(X.shape)

(15398, 1)
(15398, 1)
(15398, 29)
(15398, 24)


In [227]:
X_train, X_test, y_int_train, y_int_test = train_test_split(
            X, y_interested, test_size=0.2, stratify=y_interested, random_state=42
        )
print(X_train.shape)
print(X_test.shape)
print(y_int_train.shape)
print(y_int_test.shape)

(12318, 24)
(3080, 24)
(12318, 1)
(3080, 1)


In [228]:
y_not_int_train = y_not_interested.iloc[X_train.index.intersection(train_df.index)]
y_not_int_test = y_not_interested.iloc[X_test.index.intersection(train_df.index)]
print(y_not_int_train.shape)
print(y_not_int_test.shape)

(12318, 1)
(3080, 1)


In [229]:

xgb_model = xgb.XGBClassifier(
            objective='binary:logistic',
            learning_rate=0.03,
            max_depth=6,
            n_estimators=200,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=42,
            eval_metric='logloss'
        )

In [230]:
xgb_model.fit(X_train,y_int_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [231]:
pred_interested = xgb_model.predict(X_test)

In [232]:
accuracy = accuracy_score(pred_interested,y_int_test)
print(accuracy)

0.7668831168831168


In [233]:
y_interested_arr = np.array(y_interested['interested'])
y_interested_arr

array([0, 0, 1, ..., 0, 0, 1])

In [234]:
print(np.array_equal(pred_interested,np.array(y_int_test['interested'])))

False


In [235]:
xgb_model.fit(X_train,y_not_int_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [236]:
pred_not_interested = xgb_model.predict(X_test)

In [237]:
accuracy = accuracy_score(pred_not_interested,y_not_int_test)
print(accuracy)
print(np.array_equal(pred_not_interested,np.array(y_not_int_test['not_interested'])))

0.9756493506493507
False


In [238]:
print(X_train.shape)
X_train.info()

(12318, 24)
<class 'pandas.core.frame.DataFrame'>
Index: 12318 entries, 13715 to 13289
Data columns (total 24 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   invited                             12318 non-null  int64  
 1   is_same_city                        12318 non-null  int64  
 2   is_same_country                     12318 non-null  int64  
 3   is_same_state                       12318 non-null  int64  
 4   hrs_to_event                        12318 non-null  float64
 5   hrs_join_to_event                   12318 non-null  float64
 6   Age                                 12318 non-null  int64  
 7   pref_sunday                         12318 non-null  float64
 8   pref_monday                         12318 non-null  float64
 9   pref_tuesday                        12318 non-null  float64
 10  pref_wednesday                      12318 non-null  float64
 11  pref_thursday                 

In [239]:
X_train_nn_Scaled=[]
y_int_train_scaled=[]


In [240]:
from sklearn.preprocessing import StandardScaler,MinMaxScaler
min_max_Scaler = MinMaxScaler()
X_train_nn_Scaled = min_max_Scaler.fit_transform(X_train)
print(X_train_nn_Scaled.shape)
y_int_train_scaled = min_max_Scaler.fit_transform(y_int_train)
print(y_int_train_scaled.shape)



(12318, 24)
(12318, 1)


In [241]:
X_train_nn_Scaled.shape[1]

24

In [247]:
X_test_nn_Scaled = min_max_Scaler.fit_transform(X_test)
print(X_test_nn_Scaled.shape)
y_int_test_scaled = min_max_Scaler.fit_transform(y_int_test)
print(y_int_test_scaled.shape)

(3080, 24)
(3080, 1)


In [251]:
y_not_int_train_scaled = min_max_Scaler.fit_transform(y_not_int_train)
print(y_not_int_train_scaled.shape)
y_not_int_test_scaled = min_max_Scaler.fit_transform(y_not_int_test)
print(y_not_int_test_scaled.shape)

(12318, 1)
(3080, 1)


In [242]:
model = Sequential([
    Input(shape=(X_train_nn_Scaled.shape[1],)),
    Dense(64, activation='relu', input_shape=(X_train_nn_Scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [243]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [244]:
print(f"\nModel Architecture:")
model.summary()


Model Architecture:


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                      │ (None, 64)                  │           1,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,225 (16.50 KB)

 Trainable params: 4,225 (16.50 KB)

 Non-trainable params: 0 (0.00 B)

In [249]:
# Train the model
print(f"\nTraining the model...")
history = model.fit(
    X_train_nn_Scaled, y_int_train_scaled,
    epochs=50,
    batch_size=32,
    verbose=1
)


Training the model...
Epoch 1/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7661 - loss: 0.5010
Epoch 2/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7594 - loss: 0.4979
Epoch 3/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7630 - loss: 0.4981
Epoch 4/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7622 - loss: 0.5008
Epoch 5/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7637 - loss: 0.4961
Epoch 6/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7650 - loss: 0.4950
Epoch 7/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7616 - loss: 0.4941
Epoch 8/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7663 - loss: 0.4981
Epoch 9/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7650 - loss: 0.4910
Epoch 10/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7630 - loss: 0.5045
Epoch 11/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7624 - loss: 0.4965
Epoch 12/50
385/385 ━━━━━━━━━

In [250]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_nn_Scaled, y_int_test_scaled, verbose=0)
print(f"\nModel Performance:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Model Performance:
Test Loss: 0.5389
Test Accuracy: 0.7419


In [252]:
# Train the model
print(f"\nTraining the model...")
history = model.fit(
    X_train_nn_Scaled, y_not_int_train_scaled,
    epochs=50,
    batch_size=32,
    verbose=1
)


Training the model...
Epoch 1/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9468 - loss: 0.2210
Epoch 2/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9669 - loss: 0.1377
Epoch 3/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9640 - loss: 0.1405
Epoch 4/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9679 - loss: 0.1268
Epoch 5/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9670 - loss: 0.1252
Epoch 6/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9664 - loss: 0.1263
Epoch 7/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9655 - loss: 0.1251
Epoch 8/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9662 - loss: 0.1241
Epoch 9/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9696 - loss: 0.1127
Epoch 10/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9658 - loss: 0.1233
Epoch 11/50
385/385 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9691 - loss: 0.1117
Epoch 12/50
385/385 ━━━━━━━━━

In [253]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_nn_Scaled, y_not_int_test_scaled, verbose=0)
print(f"\nModel Performance:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


Model Performance:
Test Loss: 0.1121
Test Accuracy: 0.9747
